# Exploratory data analysis of the OTTO sample

Follows `_doc/eda_guide.md` step by step. One section per step; the decision log is `eda/eda_log.md`, comparison tables are in `eda/tables/`, figures in `eda/figures/`. Each step has its own GitHub issue (umbrella #5; step N is issue #91 + N). Rerun everything with `EDA_FORCE=1` (otherwise expensive tables are cached in `eda/tables/`).

**Two tables are analysed.**
- **Events** (`data/sample/events.parquet`): one row per event (`session`, `aid`, `ts`, `type`). No target. Used by the steps that do not need one.
- **Candidates** (`data/features/train_asof11.parquet`): one row per (session, candidate item) from the shared pool, with 18 numeric features, the category and the label. This is the modelling table for the steps that need a target.

In [ ]:
import platform
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from eda_utils import SEED, cached, log_step, save_table
from split import DAY_MS, START_MS, day_to_ms

np.random.seed(SEED)
DATA = ROOT / "data"
D14, D16 = day_to_ms(14), day_to_ms(16)
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
dev = events.filter(pl.col("ts") < D14)          # development set: every target-aware cell reads only this
CAND = DATA / "features" / "train_asof11.parquet"
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_width_chars(200)

## Step 0. Setup, framing and holdout
*Kind: diagnostic. Issue #91.*

#### Why this step exists here
Before touching any data we need to say what the analysis is for, what one row is, what the target is and how the model will be used. Everything later depends on this: the metric, the split and which columns may be used. The holdout is set aside now, so that nothing target-aware can leak into the decisions that follow.

#### Options considered
- **Holdout scheme:** time-ordered (latest rows held out), grouped by entity (session), or random/stratified. The guide's rule is: time-ordered if the model scores future data, grouped if entities repeat, random otherwise.
- **Target definition:** the guide needs one target column. Candidates: `label > 0` (a candidate is found later), a single event type (click), or the graded label. `label > 0` is the broadest and has the most positives.

In [ ]:
framing = pl.DataFrame([
    ("Decision supported", "Rank candidate items for a session prefix and a query category (search and ranking on e-commerce sessions).", "given (project goal)"),
    ("Unit of analysis", "Events table: one event. Candidates table: one (session, candidate item) pair. The entity is the session; the same session has many rows in both.", "given"),
    ("Target", "`label > 0` on the candidates table: the candidate is clicked, carted or ordered later in the same session (graded: 1 click, 2 cart, 3 order). Defined in issues #10 and #22.", "assumed for the guide (the guide needs a single target column)"),
    ("Task type", "Binary classification on candidate rows (PR-AUC), and ranking (NDCG@10) for the final metric.", "assumed"),
    ("When is the target known", "After the session prefix: the held-out events that follow the cut.", "given"),
    ("Metric and error cost", "NDCG@10 with bootstrap intervals for ranking; PR-AUC for the classification view. Cost of false positives and false negatives not given.", "assumed"),
    ("Time order / future scoring", "Yes. Timestamps exist and the model scores later sessions.", "given"),
    ("Known at prediction time", "Features are computed as of a cutoff before the session (issue #22); the label is not.", "given"),
    ("Constraints", "8 GB RAM Mac, minimise cost, no PII (anonymous ids).", "given"),
], schema=["question", "answer", "source"], orient="row")
print(framing)

# Holdout: the evaluation window (sessions that start on days 14-15), time ordered because the model scores future data.
first = events.group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
holdout = first.filter((pl.col("first_ts") >= D14) & (pl.col("first_ts") < D16))
dev_sessions = first.filter(pl.col("first_ts") < D14)
dev_max_ts = dev.select(pl.col("ts").max()).collect()[0, 0]
assert dev_max_ts < D14, "the development set must contain only events before day 14"
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())
cand_sessions = pl.read_parquet(CAND, columns=["session"])["session"].unique()
cand_first = first.filter(pl.col("session").is_in(cand_sessions.to_list()))
assert (cand_first["first_ts"] < D14).all() and (cand_first["first_ts"] >= day_to_ms(11)).all()
import lightgbm
import scipy
import sklearn

versions = {"python": platform.python_version(), "polars": pl.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "lightgbm": lightgbm.__version__, "seed": SEED}
print(f"holdout: {holdout.height:,} sessions starting on days 14-15 (time-ordered); development: {dev_sessions.height:,} sessions, events before day 14")
print("candidates table sessions (all development):", cand_sessions.len())
print(versions)

In [ ]:
e0 = log_step(0, "Setup and framing", "Diagnostic",
    "Framing table above (each answer marked given or assumed). Holdout scheme options: time-ordered, grouped by entity, random. Layout eda/{tables,figures}, seed 42, versions printed.",
    f"Development events: ts < day 14 ({dev_sessions.height:,} sessions). Holdout: {holdout.height:,} sessions that start on days 14-15. No session is on both sides. The candidates table has {cand_sessions.len():,} sessions, all in development (they start on days 11-13).",
    "Time-ordered holdout = the evaluation window of issue #6 (days 14-15). Everything target-aware reads only events before day 14.",
    "The model scores future sessions, so the latest rows are held out (guide rule). Grouping by session is implied: holdout sessions start after the development window.",
    "Random split: rejected, it would mix time periods. Group-only split: rejected, it does not test future data.",
    "High", "Steps 12 and 15 re-check the scheme; the holdout is never read before the modelling phase.",
    "yes: the target and task type are assumed from issues #10/#22. Please confirm `label > 0` is the target to use for the guide.")
print(e0)

#### Chosen approach and rationale
**Decision.** Time-ordered holdout = the evaluation window of issue #6 (days 14-15). Everything target-aware reads only events before day 14.

**Why.** The model scores future sessions, so the latest rows are held out (guide rule). Grouping by session is implied: holdout sessions start after the development window.

**Rejected.** Random split: rejected, it would mix time periods. Group-only split: rejected, it does not test future data.

**Confidence.** High

#### Interpretation: what we understand from the results
The holdout and the development set are cleanly separated in time and by session: no session is on both sides, and the 40,000 sessions of the candidate table all start on days 11-13. Two assumptions carry weight: the guide needs a single target column, and we chose `label > 0` on the candidate table from #22, keeping the graded labels for the ranking metric. This is the one required input the guide asks a human to supply, so it is marked **assumed** and needs your confirmation.

#### Impact on next steps
Steps 12 and 15 re-check the scheme; the holdout is never read before the modelling phase.

#### Out of scope
- Confirmation of the target and task type (issue #91 stays open until then).
- NDCG@10 as the final ranking metric comes from the project plan, not from this guide.

## Step 1. Load and first look
*Kind: diagnostic. Issue #92.* Options: A manual inspection, B per-column overview, C automated profile.

#### Why this step exists here
The first look tells us the size and shape of the data and shows obvious problems before any analysis. It comes right after the framing because we now know which table and which rows (development only) to look at.

#### Options considered
- **A. Manual inspection:** shape, memory, dtypes, first, last and random rows, summary statistics.
- **B. Per-column overview:** one table with dtype, missing share, unique count, example, minimum and maximum for every column.
- **C. Automated profile:** a profiling report in minimal mode. **Not run:** ydata-profiling needs numpy < 2.2 and matplotlib <= 3.10; this project runs numpy 2.4.6 and matplotlib 3.11.2, so installing it would downgrade core libraries under every earlier result. With only 4 ID-like columns, A and B already show what it would.

In [ ]:
ev = dev.collect()                                          # 13.06 million rows x 4 columns: fits in memory
print("A. shape:", ev.shape, " estimated memory:", round(ev.estimated_size() / 1e6), "MB")
print("dtypes:", dict(ev.schema))
print("first 5:\n", ev.head(5)); print("last 5:\n", ev.tail(5)); print("random 5:\n", ev.sample(5, seed=SEED))
print(ev.describe())

In [ ]:
overview = pl.DataFrame([{
    "column": c, "dtype": str(ev.schema[c]), "missing_share": ev[c].null_count() / ev.height, "n_unique": ev[c].n_unique(),
    "unique_share": ev[c].n_unique() / ev.height, "example": str(ev[c][0]), "min": str(ev[c].min()), "max": str(ev[c].max()),
} for c in ev.columns])
save_table("step01_column_overview", overview)
print("B. per-column overview:"); print(overview)
print("C. automated profile: not run, see the log (ydata-profiling needs numpy<2.2 and matplotlib<=3.10; the project has numpy",
      np.__version__, ")")

In [ ]:
issues1 = pl.DataFrame([
    ("session", "ID-like: 1.0 million distinct ids, an identifier not a feature", "exclude from features; use as the entity for splits (Steps 4, 15)"),
    ("aid", "ID-like item identifier with 1.4 million distinct values, long tail", "never a numeric feature; encoding options in Step 13"),
    ("ts", "Unix milliseconds stored as int64", "convert to a datetime for time steps (Step 2A)"),
    ("type", "int8 code 0/1/2 for click/cart/order, low cardinality", "treat as categorical (Step 2A)"),
    ("all", "No missing values, no constant column", "Step 5 has little to treat"),
], schema=["column", "problem", "handled_in"], orient="row")
save_table("step01_issue_list", issues1); print(issues1)
assert ev.null_count().sum_horizontal()[0] == 0
e1 = log_step(1, "Load and first look", "Diagnostic",
    "A manual inspection (shape, dtypes, first/last/random rows, describe); B per-column overview table; C automated profile SKIPPED: ydata-profiling requires numpy<2.2 and matplotlib<=3.10, the project uses numpy 2.4.6 and matplotlib 3.11.2, so installing it would downgrade core libraries for every earlier result; with 4 ID-like columns A and B already show its alerts.",
    f"{ev.height:,} rows x 4 columns (session int64, aid int32, ts int64, type int8), {ev.estimated_size() / 1e6:.0f} MB; no missing values; no constant column; session and aid are ID-like; ts is epoch milliseconds; type has 3 codes.",
    "Prioritised issue list saved to eda/tables/step01_issue_list.csv: convert ts, treat type as categorical, exclude ids from features.",
    "The three options were meant to be combined into one list; two were run and their findings agree.",
    "C profile: skipped for the reason above; installing at the cost of downgrading numpy was rejected.",
    "High", "Step 2 converts types; Step 13 handles aid as high-cardinality.")
print(e1)

#### Chosen approach and rationale
**Decision.** Prioritised issue list saved to eda/tables/step01_issue_list.csv: convert ts, treat type as categorical, exclude ids from features.

**Why.** The three options were meant to be combined into one list; two were run and their findings agree.

**Rejected.** C profile: skipped for the reason above; installing at the cost of downgrading numpy was rejected.

**Confidence.** High

#### Interpretation: what we understand from the results
The events table is small and clean: 4 columns, no missing values, no constant column, no impossible structure. What matters is what the columns mean: two are identifiers (session, item) that must never be used as numeric features, one is epoch time, and one is a 3-code event type. The interesting problems are not in data quality but in the structure of the candidate table, which the later steps examine.

#### Impact on next steps
Step 2 converts types; Step 13 handles aid as high-cardinality.

#### Out of scope
- Automated profile (option C): blocked by the dependency conflict above. No follow-up needed unless the numpy pin changes.

## Step 2. Types, validity and consistency
*Kind: diagnostic then fixes. Issue #93.* Options: A dtype vs meaning, B sentinel values, C range and domain rules, D cross-field consistency (development events, ts < day 14).

#### Why this step exists here
Now that we know the columns, we check that each one means what its name says and holds only possible values. Impossible values are handled by converting them to missing plus a flag, never by deleting rows. This comes before duplicates and missing values because those steps assume the values are valid.

#### Options considered
- **A. Dtype vs meaning:** numbers stored as text, dates as text, low-cardinality codes that are really categories.
- **B. Sentinel and placeholder values:** -1, 0, 999, 9999, 99999, and similar acting as missing.
- **C. Range and domain rules:** timestamps inside the 4 labelled weeks, event type in {0, 1, 2}, non-negative ids.
- **D. Cross-field consistency:** timestamps non-decreasing inside a session, a session in one block, carts after clicks and orders after carts.

In [ ]:
START, END = START_MS, START_MS + 28 * DAY_MS
n = ev.height
# A dtype vs meaning
A = pl.DataFrame([
    ("session", "int64", "identifier", "keep as id, never a numeric feature"),
    ("aid", "int32", "identifier", "keep as id"),
    ("ts", "int64", "epoch milliseconds", "convert to datetime for time steps; the raw integer is kept for splits"),
    ("type", "int8", "categorical code 0 click, 1 cart, 2 order", "cast to a categorical for encoders; the code is fine for trees"),
], schema=["column", "stored_as", "meaning", "conversion"], orient="row"); print("A.\n", A)
# B sentinels
sent = [-1, 0, 999, 9999, 99999]
hits = [{"column": c, "value": v, "rows": int((ev[c].cast(pl.Int64) == v).sum())} for c in ("session", "aid", "ts", "type") for v in sent]
B = pl.DataFrame([h for h in hits if h["rows"] > 0] or [{"column": "-", "value": 0, "rows": 0}]).with_columns((pl.col("rows") / n).alias("share")); print("B. sentinel-like values found:\n", B)
# C ranges
C = pl.DataFrame([
    ("ts inside the 4 labelled weeks", int(((ev["ts"] < START) | (ev["ts"] >= END)).sum())),
    ("type not in {0,1,2}", int((~ev["type"].is_in([0, 1, 2])).sum())),
    ("aid < 0", int((ev["aid"] < 0).sum())), ("session < 0", int((ev["session"] < 0).sum())),
    ("ts in the future of the development cut (>= day 14)", int((ev["ts"] >= D14).sum())),
], schema=["rule", "violations"], orient="row").with_columns((pl.col("violations") / n).alias("share")); print("C.\n", C)
# D cross-field
srt = ev.select(pl.col("session"), pl.col("ts"), (pl.col("ts").diff() < 0).alias("backwards"), (pl.col("session") != pl.col("session").shift(1)).alias("new"))
backwards_within = int((srt["backwards"] & ~srt["new"]).sum())
blocks = int(srt["new"].sum()) + 1; sessions = ev["session"].n_unique()   # +1: the first row has no previous row
first_seen = ev.group_by("session", "aid").agg(
    pl.col("ts").filter(pl.col("type") == 0).min().alias("t_click"), pl.col("ts").filter(pl.col("type") == 1).min().alias("t_cart"),
    pl.col("ts").filter(pl.col("type") == 2).min().alias("t_order"))
cart_no_click = first_seen.filter(pl.col("t_cart").is_not_null() & (pl.col("t_click").is_null() | (pl.col("t_cart") < pl.col("t_click")))).height
order_no_cart = first_seen.filter(pl.col("t_order").is_not_null() & (pl.col("t_cart").is_null() | (pl.col("t_order") < pl.col("t_cart")))).height
pairs = first_seen.height
D = pl.DataFrame([
    ("timestamps go backwards inside a session", backwards_within, backwards_within / n),
    ("a session appears in more than one block (blocks - sessions)", blocks - sessions, (blocks - sessions) / n),
    ("a cart of an item with no earlier click (share of session-item pairs)", cart_no_click, cart_no_click / pairs),
    ("an order of an item with no earlier cart (share of session-item pairs)", order_no_cart, order_no_cart / pairs),
], schema=["rule", "violations", "share"], orient="row"); print("D.\n", D)
for name, t in (("A", A), ("B", B), ("C", C), ("D", D)):
    save_table(f"step02_{name}", t)

In [ ]:
bad_share = max(float(C["share"].max()), float(D["share"][:2].max()))
e2 = log_step(2, "Types, validity and consistency", "Diagnostic then fixes",
    "A dtype vs meaning; B sentinel-like values (-1, 0, 999, 9999, 99999); C range and domain rules; D cross-field consistency (ordering inside sessions, cart/order after click/cart). All four run on the development events.",
    f"A: 4 columns, all correctly typed except that ts is epoch ms. B: no placeholder found. The scan flags aid values 0, 999, 9999 and 99999 (2, 4, 44 and 1 events), but item ids are dense integers from 0 to 1,855,602, so these are ordinary ids; session 0 (86 events) is the first session; type 0 is the code for a click (89.8% of events). Details in eda/tables/step02_B.csv. C: {int(C['violations'].sum()):,} range violations (ts inside the 4 weeks, type in {{0,1,2}}, ids non-negative). D: {backwards_within:,} backwards timestamps inside sessions; {cart_no_click / pairs:.1%} of carted items had no earlier click in the same session and {order_no_cart / pairs:.1%} of ordered items had no earlier cart.",
    "No fixes needed for A-C. D is a property of the data (users can add to the cart from a page without a logged click), not an error. The sentinel-like values are not placeholders (evidence above), so nothing goes on the human question list.",
    "Rule: apply unambiguous fixes, convert impossible values to missing with a flag, never delete rows. There was nothing impossible.",
    "Nothing to reject: all impossible-value options found no violations. Treating cart-without-click as an error was rejected because it is common and plausible.",
    "High", "Step 4 uses the key (session, aid, ts, type); Step 12 must not assume a click precedes a cart.")
print(e2)

#### Chosen approach and rationale
**Decision.** No fixes needed for A-C. D is a property of the data (users can add to the cart from a page without a logged click), not an error. The sentinel-like values are not placeholders (evidence above), so nothing goes on the human question list.

**Why.** Rule: apply unambiguous fixes, convert impossible values to missing with a flag, never delete rows. There was nothing impossible.

**Rejected.** Nothing to reject: all impossible-value options found no violations. Treating cart-without-click as an error was rejected because it is common and plausible.

**Confidence.** High

#### Interpretation: what we understand from the results
Nothing is broken: every value is in range and time order inside sessions is intact. The values that look like sentinels (item ids 0, 999, 9999, 99999) are ordinary ids, because item ids are dense integers from 0 to 1,855,602; session 0 is the first session; type 0 is a click. The one apparent inconsistency, carts of items never clicked (2.1% of session-item pairs) and orders with no cart (0.3%), is normal shop behaviour (adding to the cart from a list page). It matters later because a click cannot be assumed to precede a cart.

#### Impact on next steps
Step 4 uses the key (session, aid, ts, type); Step 12 must not assume a click precedes a cart.

#### Out of scope
- None.

## Step 3. Target analysis
*Kind: diagnostic. Issue #94.* Target on the candidates table: `label > 0`.

#### Why this step exists here
We look at the target before any feature, because the target's shape decides the metric (PR-AUC or ROC-AUC), the split and how much we can trust per-class numbers. This step uses the candidate table, which is the only table with a target.

#### Options considered
- **A. Distribution:** class counts and shares, the imbalance ratio.
- **B. Absolute minority count:** the number of rows in the rarest class, not only its share (a small share of a huge table can still be plenty).
- **C. Stability:** the target rate by day and by the largest segments.
- **D. Missing target:** rows with no label.

In [ ]:
cand = pl.read_parquet(CAND, columns=["session", "aid", "category", "label", "y_click", "y_cart", "y_order"])
print("A. label counts and shares:")
A3 = cand.group_by("label").len().sort("label").with_columns((pl.col("len") / cand.height).alias("share")); print(A3)
pos = int((cand["label"] > 0).sum())
print(f"y = label > 0: {pos:,} positives of {cand.height:,} rows ({pos / cand.height:.3%}); imbalance ratio 1 : {(cand.height - pos) / pos:.0f}")
B3 = pl.DataFrame({"class": ["any (label>0)", "click", "cart", "order"], "positives": [pos, int(cand["y_click"].sum()), int(cand["y_cart"].sum()), int(cand["y_order"].sum())]}); print("B. absolute minority counts:"); print(B3)
first_ts = first.filter(pl.col("session").is_in(cand["session"].unique().to_list()))
c2 = cand.join(first_ts, on="session").with_columns(((pl.col("first_ts") - day_to_ms(0)) // DAY_MS).alias("day"))
C3a = c2.group_by("day").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("day"); print("C. by day of the session start:"); print(C3a)
top5 = c2.group_by("category").len().sort("len", descending=True).head(5)["category"].to_list()
C3b = c2.filter(pl.col("category").is_in(top5)).group_by("category").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("rows", descending=True); print("C. by the 5 largest categories:"); print(C3b)
print("D. rows without a target:", int(cand["label"].null_count()))
for nme, t in (("A", A3), ("B", B3), ("C_day", C3a), ("C_category", C3b)):
    save_table(f"step03_{nme}", t)

In [ ]:
# minority counts in the development set
dev_pos = pos  # the candidates table contains development sessions only
e3 = log_step(3, "Target analysis", "Diagnostic",
    "A class counts and shares; B absolute minority counts; C target rate by day and by the 5 largest categories; D missing target.",
    f"{cand.height:,} rows, {pos:,} positives ({pos / cand.height:.3%}); ratio 1:{(cand.height - pos) / pos:.0f}. Positives by type: click {int(cand['y_click'].sum()):,}, cart {int(cand['y_cart'].sum()):,}, order {int(cand['y_order'].sum()):,}. Target rate by start day and by category in eda/tables/step03_C_*.csv. No missing target.",
    "Class imbalance is SEVERE (minority under 1 percent): Step 14 handles it. The cart and order classes are small but above 100 rows; orders are the rarest.",
    "Guide rules: flag imbalance under 10 percent, severe under 1 percent; remove rows with no target (none).",
    "No option was rejected; all four were run.",
    "High", "Step 14 must use PR-AUC; Step 15 uses grouped folds because a session has 200 rows; the order task has few positives, so per-task results are noisy.")
print(e3)

#### Chosen approach and rationale
**Decision.** Class imbalance is SEVERE (minority under 1 percent): Step 14 handles it. The cart and order classes are small but above 100 rows; orders are the rarest.

**Why.** Guide rules: flag imbalance under 10 percent, severe under 1 percent; remove rows with no target (none).

**Rejected.** No option was rejected; all four were run.

**Confidence.** High

#### Interpretation: what we understand from the results
The target is severely imbalanced (0.44% positives, 1 in 227), which makes ROC-AUC look flattering and PR-AUC the honest metric. It is stable over the three days and across the five largest categories (0.35% to 0.51%), so there is no sign of a drifting definition. The click class is large (34,759 rows), while carts (2,512) and orders (589) are small but above the 100-row threshold, so per-task numbers will be noisy. That explains the wide gap between the validation and evaluation AUC of the order task seen in issue #28.

#### Impact on next steps
Step 14 must use PR-AUC; Step 15 uses grouped folds because a session has 200 rows; the order task has few positives, so per-task results are noisy.

#### Out of scope
- None.

## Step 4. Duplicates
*Kind: selection (how to treat them). Issue #95.* Detection: A exact, B key-based, C near duplicates, D conflicting labels.

#### Why this step exists here
Repeated rows can inflate counts, bias popularity and co-visitation, and leak across a split. We look for them before missing values and features, because a duplicate is either an error to drop or a real repeated observation to keep, and that decision changes every count downstream.

#### Options considered
Detection options (all run): **A. exact** (all columns equal); **B. key-based** (same session, item and time, with or without the type); **C. near duplicates** (same session, item and type within one second); **D. conflicting labels** (identical features, different target). Treatment options: keep everything, drop exact duplicates keeping the first, drop only click and cart duplicates, aggregate per key.

In [ ]:
nrows = ev.height
# A exact
A4 = nrows - ev.unique().height
# B key based (session, aid, ts) and (session, aid, type, ts)
B1 = nrows - ev.select("session", "aid", "ts").unique().height
B2 = nrows - ev.select("session", "aid", "type", "ts").unique().height
# C near duplicates: same session, item and type within the same second
C4 = nrows - ev.with_columns((pl.col("ts") // 1000).alias("sec")).select("session", "aid", "type", "sec").unique().height
dup = pl.DataFrame([("A exact (all 4 columns)", A4), ("B key (session, aid, type, ts)", B2), ("B key (session, aid, ts), any type", B1), ("C near (session, aid, type, same second)", C4)],
    schema=["kind", "rows_beyond_the_first"], orient="row").with_columns((pl.col("rows_beyond_the_first") / nrows).alias("share")); print(dup)
save_table("step04_events_duplicates", dup)
# how do the exact duplicates cluster?
exact = ev.group_by(ev.columns).len().filter(pl.col("len") > 1)
print("groups with exact duplicates:", exact.height, "; largest group size:", int(exact["len"].max()) if exact.height else 0)
by_type = ev.filter(pl.struct(ev.columns).is_duplicated()).group_by("type").len().sort("type"); print("exact duplicates by event type (all copies):"); print(by_type)
# across the development / holdout boundary: sessions on both sides (none, checked in Step 0)
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())

In [ ]:
# D conflicting labels on the candidates table: identical (rounded) features but different target
feat = pl.read_parquet(CAND).drop(["session", "aid"])
num = [c for c in feat.columns if feat.schema[c] in (pl.Float32, pl.Float64)]
rounded = feat.with_columns([pl.col(c).round(3) for c in num])
keys = [c for c in rounded.columns if c not in ("label", "y_click", "y_cart", "y_order")]
groups = rounded.with_columns((pl.col("label") > 0).alias("y")).group_by(keys).agg(pl.len().alias("n"), pl.col("y").n_unique().alias("n_targets"), pl.col("y").sum().alias("pos"))
multi = groups.filter(pl.col("n") > 1)
conflict = multi.filter(pl.col("n_targets") > 1)
rows_in_conflict = int(conflict["n"].sum()); minority_in_conflict = int(conflict["pos"].sum())
print(f"D. feature combinations shared by 2+ rows: {multi.height:,} ({int(multi['n'].sum()):,} rows); with conflicting targets: {conflict.height:,} combinations, {rows_in_conflict:,} rows ({rows_in_conflict / feat.height:.2%} of all rows)")
print("share of positives that sit in a conflicting group:", round(minority_in_conflict / pos, 3))
D4 = pl.DataFrame({"measure": ["combinations shared by 2+ rows", "rows in those combinations", "combinations with conflicting targets", "rows in conflicting combinations", "share of all rows", "share of positives in a conflict"],
                   "value": [float(multi.height), float(multi["n"].sum()), float(conflict.height), float(rows_in_conflict), rows_in_conflict / feat.height, minority_in_conflict / pos]})
save_table("step04_candidates_conflicts", D4); print(D4)
del feat, rounded, groups

In [ ]:
order_dups = int(by_type.filter(pl.col("type") == 2)["len"][0]); click_dups = int(by_type.filter(pl.col("type") == 0)["len"][0]); cart_dups = int(by_type.filter(pl.col("type") == 1)["len"][0])
n_order = int((ev["type"] == 2).sum())
print(f"order events involved in exact duplicates: {order_dups:,} of {n_order:,} ({order_dups / n_order:.1%}); clicks {click_dups:,} of {int((ev['type'] == 0).sum()):,} ({click_dups / int((ev['type'] == 0).sum()):.2%}); carts {cart_dups:,}")
e4 = log_step(4, "Duplicates", "Selection (how to treat them)",
    "A exact; B key-based ((session, aid, type, ts) and (session, aid, ts)); C near-duplicates (same session, item, type in the same second); D conflicting labels on the candidates table (features rounded to 3 decimals). Treatments considered: keep everything, drop exact duplicates keeping the first, drop only the click and cart duplicates, aggregate per key.",
    f"A: {A4:,} rows ({A4 / nrows:.3%}) are exact copies of an earlier row, in {exact.height:,} groups (largest group 6). B is identical to A (no key repeats with a different value or type). C: {C4:,} rows ({C4 / nrows:.3%}). By event type, the copies are mostly ORDERS: {order_dups:,} order events ({order_dups / n_order:.1%} of all order events) are involved, against {click_dups:,} clicks and {cart_dups:,} carts. D: {conflict.height} feature combinations have conflicting targets, covering {rows_in_conflict} rows ({rows_in_conflict / cand.height:.5%} of {cand.height:,}); each holds one positive and one negative, i.e. {minority_in_conflict} positives ({minority_in_conflict / pos:.2%} of all positives). Duplicates across the development/holdout boundary: none (no session is on both sides).",
    "Drop exact duplicate CLICK and CART events, keeping the first (double logging in the same millisecond has no business reason). KEEP order duplicates: an order of the same item logged twice at the same millisecond is most likely a quantity above 1, which is real. Conflicting labels: none to treat. Not applied to the data used by the earlier issues (#7, #17-#29): the events involved are 0.10% of clicks and 0.09% of carts (copies including the first occurrence), and the effect on those results is measured in follow-up issue #109.",
    "Guide rules: exact duplicates with no business reason are dropped keeping the first; legitimate repeated observations are kept; conflicting labels are quantified and never deleted without approval. Orders have a plausible business reason (quantity), clicks and carts do not.",
    "Keep everything: rejected for clicks and carts (they would double count in co-visitation and popularity). Drop all exact duplicates: rejected because it would remove real order quantities (8.7% of order events are involved). Aggregate per key: rejected, it changes the event grain that Steps 12 and 16 rely on. Near-duplicates (C): NOT dropped: a second click on the same item within one second cannot be told from a real fast repeat without inspecting a sample, and the guide says drop near duplicates only if a sample shows they are the same record.",
    "Medium (the order-quantity reading is an inference, not something the data proves)",
    "Affects the event counts by 0.15%; co-visitation, popularity and features from earlier issues used the un-deduplicated events. Follow-up issue #109 quantifies the effect on the closed results. Step 15 uses grouped folds by session regardless.")
print(e4)

#### Chosen approach and rationale
**Decision.** Drop exact duplicate CLICK and CART events, keeping the first (double logging in the same millisecond has no business reason). KEEP order duplicates: an order of the same item logged twice at the same millisecond is most likely a quantity above 1, which is real. Conflicting labels: none to treat. Not applied to the data used by the earlier issues (#7, #17-#29): the events involved are 0.10% of clicks and 0.09% of carts (copies including the first occurrence), and the effect on those results is measured in follow-up issue #109.

**Why.** Guide rules: exact duplicates with no business reason are dropped keeping the first; legitimate repeated observations are kept; conflicting labels are quantified and never deleted without approval. Orders have a plausible business reason (quantity), clicks and carts do not.

**Rejected.** Keep everything: rejected for clicks and carts (they would double count in co-visitation and popularity). Drop all exact duplicates: rejected because it would remove real order quantities (8.7% of order events are involved). Aggregate per key: rejected, it changes the event grain that Steps 12 and 16 rely on. Near-duplicates (C): NOT dropped: a second click on the same item within one second cannot be told from a real fast repeat without inspecting a sample, and the guide says drop near duplicates only if a sample shows they are the same record.

**Confidence.** Medium (the order-quantity reading is an inference, not something the data proves)

#### Interpretation: what we understand from the results
Duplicates are rare (0.15% of events) but not random: they are concentrated in orders (8.7% of order events are involved, against 0.10% of clicks). That points to order quantities (the same item logged twice at the same millisecond) more than to logging errors, which is why order duplicates are kept. This is an inference, so the confidence is Medium. Conflicting labels are almost absent (7 feature combinations out of 8 million rows), so label noise does not put a ceiling on the accuracy we can reach with this table.

#### Impact on next steps
Affects the event counts by 0.15%; co-visitation, popularity and features from earlier issues used the un-deduplicated events. Follow-up issue #109 quantifies the effect on the closed results. Step 15 uses grouped folds by session regardless.

#### Out of scope
- The effect of the de-duplication on the closed co-visitation, popularity and feature results: follow-up issue #109.

## Step 5. Missing values
*Kind: diagnostic (5a) then selection (5b). Issue #96.*

Neither table has explicit nulls. But the candidates table (issue #22) **codes** "not available" as a fill value: `rank_covis` and `rank_pop` are 201 when the candidate was not returned by that retriever. Those fills are structural missing values, and 5a/5b treat them as such. Cross-validation is 3-fold grouped by session on a sample of 6,000 queries (1.2 million rows); two model families at defaults (logistic regression with standardisation, LightGBM); metric PR-AUC (guide section 3, rule 14).

#### Why this step exists here
Missing values change every later comparison, so we work out why values are missing before choosing a treatment (5a), and then pick the treatment by cross-validated score, not by habit (5b). It comes after duplicates and before outliers and transforms because imputation must be fitted first for those steps to see complete columns.

#### Options considered
**5a Diagnose:** counts per column; co-missingness; relation of missingness to the target, to other columns and to time; classify each column as MCAR-, MAR- or MNAR-like.
**5b Treat (numeric columns):** drop columns above 30/50/80 percent missing; drop rows with any missing; mean; median; median + missing indicator; sentinel + indicator; segment median; KNN imputation; iterative (model-based) imputation; leave missing for the gradient boosting model. **Categorical:** most frequent level or an explicit `missing` level (not applicable: `category` has no missing values).

In [ ]:
import scipy.stats as st
from eda_cv import NUM, decide, evaluate_option, load_sample, make_folds, summarise

sample = load_sample(ROOT, 6000)
print(f"CV sample: {sample['session'].n_unique():,} queries, {sample.height:,} rows, {int(sample['y'].sum()):,} positives ({sample['y'].mean():.3%}); seed {SEED}")
full = pl.read_parquet(CAND, columns=["session", "label"] + NUM + ["category"])

# 5a. Diagnose
explicit_nulls = full.null_count().to_dicts()[0]
print("A. explicit nulls in the candidates table:", {k: v for k, v in explicit_nulls.items() if v}, "(none)" if not any(explicit_nulls.values()) else "")
coded = pl.DataFrame([
    ("rank_covis", "== 201 (not returned by co-visitation)", int((full["rank_covis"] == 201).sum())),
    ("rank_pop", "== 201 (not in the category popularity list)", int((full["rank_pop"] == 201).sum())),
    ("covis_max", "== 0 (no co-visitation with the prefix; a real zero)", int((full["covis_max"] == 0).sum())),
    ("hours_since_last_seen", "filled for unseen items (item_seen == 0)", int((full["item_seen"] == 0).sum())),
], schema=["column", "coded_missing_value", "rows"], orient="row").with_columns((pl.col("rows") / full.height).alias("share"))
print("A. coded missing values (share of all rows):"); print(coded)
save_table("step05_coded_missing", coded)
y_all = (full["label"] > 0)
rel = pl.DataFrame([{"column": c, "target_rate_missing": float(y_all.filter(m).mean()), "target_rate_present": float(y_all.filter(~m).mean()),
                     "mean_prefix_len_missing": float(full.filter(m)["prefix_len"].mean()), "mean_prefix_len_present": float(full.filter(~m)["prefix_len"].mean())}
                    for c, m in (("rank_covis", full["rank_covis"] == 201), ("rank_pop", full["rank_pop"] == 201))])
print("C. relation to the target and to the prefix length:"); print(rel)
save_table("step05_relation", rel)
both = full.select((pl.col("rank_covis") == 201).alias("covis_missing"), (pl.col("rank_pop") == 201).alias("pop_missing")).group_by("covis_missing", "pop_missing").len().with_columns((pl.col("len") / full.height).alias("share"))
print("B. co-missingness (rank_covis missing x rank_pop missing):"); print(both.sort("covis_missing", "pop_missing"))
print("phi correlation of the two missingness indicators:", round(float(np.corrcoef((full["rank_covis"] == 201).to_numpy(), (full["rank_pop"] == 201).to_numpy())[0, 1]), 3))

In [ ]:
# 5b. Options under the shared protocol. Missing = rank_covis (77%) and rank_pop (12%) set to NaN.
MISS = ["rank_covis", "rank_pop"]
mi = [NUM.index(c) for c in MISS]
sample_m = sample.with_columns([pl.when(pl.col(c) == 201).then(None).otherwise(pl.col(c)).alias(c) for c in MISS])
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer


def arrays(tr, va):
    return (tr.select(NUM).to_numpy().astype(np.float32), tr["y"].to_numpy(), va.select(NUM).to_numpy().astype(np.float32), va["y"].to_numpy())


def with_fill(fill_fn, indicator=False, drop_cols=()):
    """Generic prep: fill_fn(xtr, xva, tr, va) returns filled copies; optional indicator columns; optional dropped columns."""
    def prep(tr, va):
        xtr, ytr, xva, yva = arrays(tr, va)
        mtr, mva = np.isnan(xtr[:, mi]), np.isnan(xva[:, mi])
        xtr, xva = fill_fn(xtr, xva, tr, va)
        if indicator:
            xtr, xva = np.hstack([xtr, mtr.astype(np.float32)]), np.hstack([xva, mva.astype(np.float32)])
        if drop_cols:
            keep = [i for i in range(xtr.shape[1]) if i not in {NUM.index(c) for c in drop_cols}]
            xtr, xva = xtr[:, keep], xva[:, keep]
        return xtr, ytr, xva, yva
    return prep


def stat_fill(stat):
    def fn(xtr, xva, tr, va):
        xtr, xva = xtr.copy(), xva.copy()
        for i in mi:
            v = float(stat(xtr[~np.isnan(xtr[:, i]), i]))
            xtr[np.isnan(xtr[:, i]), i] = v; xva[np.isnan(xva[:, i]), i] = v
        return xtr, xva
    return fn


def const_fill(value):
    def fn(xtr, xva, tr, va):
        xtr, xva = xtr.copy(), xva.copy()
        for i in mi:
            xtr[np.isnan(xtr[:, i]), i] = value; xva[np.isnan(xva[:, i]), i] = value
        return xtr, xva
    return fn


def segment_median(xtr, xva, tr, va):
    xtr, xva = xtr.copy(), xva.copy()
    cat_tr, cat_va = tr["category"].to_numpy(), va["category"].to_numpy()
    for i in mi:
        glob = float(np.nanmedian(xtr[:, i]))
        med = {c: float(np.nanmedian(xtr[(cat_tr == c) & ~np.isnan(xtr[:, i]), i])) if ((cat_tr == c) & ~np.isnan(xtr[:, i])).any() else glob for c in np.unique(cat_tr)}
        for x, cat in ((xtr, cat_tr), (xva, cat_va)):
            m = np.isnan(x[:, i])
            x[m, i] = [med.get(c, glob) for c in cat[m]]
    return xtr, xva


def native(tr, va):                       # leave missing: only the gradient boosting model can use NaN
    xtr, ytr, xva, yva = arrays(tr, va)
    return {"gbm": (xtr, ytr, xva, yva)}


shares = {c: float(sample_m[c].null_count() / sample_m.height) for c in MISS}
dropped_30 = [c for c in MISS if shares[c] > 0.30]; dropped_50 = [c for c in MISS if shares[c] > 0.50]; dropped_80 = [c for c in MISS if shares[c] > 0.80]
print("missing shares in the sample:", {k: round(v, 3) for k, v in shares.items()}, "-> dropped at 30%:", dropped_30, " at 50%:", dropped_50, " at 80%:", dropped_80)
opts = {
    "leave missing (native, gbm only)": native,
    "sentinel 201 + indicator": with_fill(const_fill(201.0), indicator=True),
    "sentinel 201 (no indicator)": with_fill(const_fill(201.0)),
    "mean": with_fill(stat_fill(np.mean)),
    "median": with_fill(stat_fill(np.median)),
    "median + indicator": with_fill(stat_fill(np.median), indicator=True),
    "segment median (by category)": with_fill(segment_median),
    "drop columns > 30% missing (+ median fill of the rest)": with_fill(stat_fill(np.median), drop_cols=dropped_30),
    "drop columns > 50% missing (+ median fill of the rest)": with_fill(stat_fill(np.median), drop_cols=dropped_50),
    "keep columns up to 80% missing (median fill)": with_fill(stat_fill(np.median), drop_cols=dropped_80),
}
folds = make_folds(sample_m)
def run_all():
    rows = []
    for name, prep in opts.items():
        t0 = time.time()
        rows += evaluate_option(name, prep, sample_m, folds)
        print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv5 = cached("step05_cv_rows", run_all)
sum5 = summarise(cv5.to_dicts())
print(sum5.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc", "n_features").sort("family", "pr_auc", descending=[False, True]))

In [ ]:
# drop rows with any missing: changes the sample, so it is reported but not ranked
def drop_rows_option():
    rows = []
    def prep(tr, va):
        tr, va = tr.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS)), va.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS))
        return arrays(tr, va)
    rows += evaluate_option("drop rows with any missing (not ranked)", prep, sample_m, folds)
    return pl.DataFrame(rows)
dr = cached("step05_cv_droprows", drop_rows_option)
kept = sample_m.filter(pl.all_horizontal(pl.col(c).is_not_null() for c in MISS))
print(f"drop rows: keeps {kept.height / sample_m.height:.1%} of rows and {kept['y'].sum() / sample_m['y'].sum():.1%} of the positives")
print(summarise(dr.to_dicts()).select("option", "family", "pr_auc", "pr_auc_sd", "rows_train"))

In [ ]:
# KNN and iterative imputation are expensive: run on a 500-query sample, next to two cheap options on the SAME rows
small = load_sample(ROOT, 500).with_columns([pl.when(pl.col(c) == 201).then(None).otherwise(pl.col(c)).alias(c) for c in MISS])
folds_small = make_folds(small)
def knn_fill(xtr, xva, tr, va):
    ref = xtr[np.random.default_rng(SEED).choice(len(xtr), size=min(8000, len(xtr)), replace=False)]
    k = KNNImputer(n_neighbors=5).fit(ref)
    return k.transform(xtr), k.transform(xva)
def iter_fill(xtr, xva, tr, va):
    sub = xtr[np.random.default_rng(SEED).choice(len(xtr), size=min(20000, len(xtr)), replace=False)]
    imp = IterativeImputer(max_iter=5, random_state=SEED).fit(sub)
    return imp.transform(xtr), imp.transform(xva)
small_opts = {"small sample: median + indicator": with_fill(stat_fill(np.median), indicator=True),
              "small sample: sentinel 201 + indicator": with_fill(const_fill(201.0), indicator=True),
              "small sample: KNN (k=5, 8,000 reference rows)": with_fill(knn_fill),
              "small sample: iterative (BayesianRidge)": with_fill(iter_fill)}
def run_small():
    rows = []
    for name, prep in small_opts.items():
        t0 = time.time(); rows += evaluate_option(name, prep, small, folds_small); print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
small_cv = cached("step05_cv_small", run_small)
sum_small = summarise(small_cv.to_dicts())
print(f"small sample: {small.height:,} rows, {int(small['y'].sum())} positives")
print(sum_small.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc"))

In [ ]:
# distribution check: do the imputed values resemble the observed ones? (fold 0 fit, imputed values in missing rows)
tr0, va0 = sample_m[folds[0][0]], sample_m[folds[0][1]]
xtr0, _, xva0, _ = arrays(tr0, va0)
observed = {c: xva0[~np.isnan(xva0[:, NUM.index(c)]), NUM.index(c)] for c in MISS}
rows = []
for name, fn in (("mean", stat_fill(np.mean)), ("median", stat_fill(np.median)), ("sentinel 201", const_fill(201.0)), ("segment median", segment_median)):
    _, filled = fn(xtr0, xva0, tr0, va0)
    for c in MISS:
        i = NUM.index(c); miss = np.isnan(xva0[:, i])
        rows.append({"option": name, "column": c, "imputed_mean": float(filled[miss, i].mean()), "observed_mean": float(observed[c].mean()),
                     "imputed_sd": float(filled[miss, i].std()), "observed_sd": float(observed[c].std()),
                     "ks_vs_observed": float(st.ks_2samp(filled[miss, i][:50000], observed[c][:50000]).statistic)})
dist5 = pl.DataFrame(rows); save_table("step05_distribution_check", dist5); print(dist5)

In [ ]:
complexity5 = {"leave missing (native, gbm only)": 0, "sentinel 201 (no indicator)": 1, "sentinel 201 + indicator": 1,
               "drop columns > 30% missing (+ median fill of the rest)": 1, "drop columns > 50% missing (+ median fill of the rest)": 1,
               "keep columns up to 80% missing (median fill)": 2, "mean": 2, "median": 2, "median + indicator": 2, "segment median (by category)": 3}
save_table("step05_cv_summary", sum5)
decisions5 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum5, fam, complexity5)
    decisions5[fam] = (w, tied, info)
    print(f"first pass, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied within noise: {len(tied)} options; winner by simplicity: {w}")

# Guide rule 6: for a close call, repeat the cross-validation with different fold seeds. Every option tied within noise at the first pass, 2 extra seeds.
top = set(decisions5["linear"][1]) | set(decisions5["gbm"][1])      # every option tied with the best at the first pass, in either family
def rerun5():
    rows = []
    for seed in (43, 44):
        f = make_folds(sample_m, seed=seed)
        for name in sorted(top):
            for r in evaluate_option(name, opts[name], sample_m, f):
                rows.append({**r, "fold": r["fold"] + 3 * (seed - 42)})
    return pl.DataFrame(rows)
rep5 = cached("step05_cv_repeat", rerun5)
all5 = pl.concat([cv5.filter(pl.col("option").is_in(sorted(top))), rep5.select(cv5.columns)])
sum5r = summarise(all5.to_dicts()); save_table("step05_cv_summary_repeated", sum5r)
print("repeated CV (3 fold seeds x 3 folds = 9 folds) for the top options:"); print(sum5r.select("option", "family", "pr_auc", "pr_auc_sd", "folds").sort("family", "pr_auc", descending=[False, True]))
final5 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum5r, fam, complexity5)
    final5[fam] = (w, tied, info)
    print(f"after repeating, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {tied}; winner by simplicity: {w}")

In [ ]:
wl, tl, il = final5["linear"]; wg, tg, ig = final5["gbm"]
rc = rel.filter(pl.col("column") == "rank_covis").row(0, named=True); rp = rel.filter(pl.col("column") == "rank_pop").row(0, named=True)
knn_l = sum_small.filter((pl.col("option").str.contains("KNN")) & (pl.col("family") == "linear"))["pr_auc"][0]; med_l = sum_small.filter((pl.col("option").str.contains("median")) & (pl.col("family") == "linear"))["pr_auc"][0]
it_l = sum_small.filter((pl.col("option").str.contains("iterative")) & (pl.col("family") == "linear"))["pr_auc"][0]
e5 = log_step(5, "Missing values", "Diagnostic (5a) then Selection (5b)",
    "5a: A counts, B co-missingness, C relation to target/prefix length. 5b (10 options, both families, 3-fold grouped CV, 6,000 queries): leave missing (native GBM), sentinel 201 with and without indicator, mean, median, median + indicator, segment median, drop columns above 30% / 50% missing, keep up to 80% (median fill); drop rows (reported, not ranked); KNN (k=5) and iterative imputation (on a 500-query sample, next to two cheap options on the same rows). Close calls repeated with 2 extra fold seeds. Categorical fills: not applicable (category has no missing values).",
    f"No explicit nulls in either table. The candidates table CODES missing values as 201: rank_covis in {coded['share'][0]:.1%} of rows and rank_pop in {coded['share'][1]:.1%}. Both are informative: target rate {rc['target_rate_missing']:.2%} when rank_covis is missing vs {rc['target_rate_present']:.2%} present; {rp['target_rate_missing']:.2%} vs {rp['target_rate_present']:.2%} for rank_pop. The two are almost mutually exclusive (phi -0.685): a candidate comes from the co-visitation list, the popularity list, or both, so it is missing from at most one. Classification: MNAR-like (informative, structural). Linear model: median + indicator, sentinel + indicator and sentinel alone are tied at PR-AUC about 0.048; mean, median and dropping the column are lower (0.042-0.044). Gradient boosting: all options within noise (PR-AUC 0.034-0.039). KNN and iterative imputation do not beat median + indicator on the small sample (linear {knn_l:.3f} and {it_l:.3f} vs {med_l:.3f}). Dropping rows keeps only 11.1% of rows and 44.4% of positives (PR-AUC not comparable). No imputed distribution matches the observed one (KS 0.40-1.0), which is expected when the value means 'not in the list'.",
    f"Linear model: {wl}. Gradient boosting: {wg}. One variant per model family in the preprocessing spec.",
    f"Guide rules: options within the larger fold standard deviation of the best are tied (linear: {len(tl)} options, gbm: {len(tg)} options after repeating the CV with 3 fold seeds); among tied options take the simplest (leave as is < fixed rule < learned statistic < model). Missingness is informative, so an option that keeps the information (an indicator, or a sentinel the trees can split on) is preferred.",
    "Drop columns above 30% or 50%: rejected, it removes the strongest missingness signal (rank_covis is missing for 77%), PR-AUC 0.042 (linear) and 0.034 (gbm). Keeping columns up to 80%: identical to the median fill (no column is above 80%). Mean/median/segment median: lower for the linear model because they erase the fact that the candidate was not returned. KNN and iterative: no better than the simple options and far more expensive (KNN took 86 s on 100,000 rows). Drop rows: rejected, it discards 89% of rows and 56% of positives and would bias the sample.",
    "Linear: Medium (three tied options, chosen by the simplicity rule). Gradient boosting: Low (all options tied: the choice does not matter much, the tie-break picked the simplest). The gbm choice goes on the needs-human-review list.",
    "Step 6-7 use the coded values as they are; Step 14 keeps in mind that most negatives have rank_covis = 201, so one code carries much of the signal. The two coded columns must be treated the same in training and serving (issue #58).")
print(e5)

#### Chosen approach and rationale
**Decision.** Linear model: sentinel 201 + indicator. Gradient boosting: leave missing (native, gbm only). One variant per model family in the preprocessing spec.

**Why.** Guide rules: options within the larger fold standard deviation of the best are tied (linear: 3 options, gbm: 8 options after repeating the CV with 3 fold seeds); among tied options take the simplest (leave as is < fixed rule < learned statistic < model). Missingness is informative, so an option that keeps the information (an indicator, or a sentinel the trees can split on) is preferred.

**Rejected.** Drop columns above 30% or 50%: rejected, it removes the strongest missingness signal (rank_covis is missing for 77%), PR-AUC 0.042 (linear) and 0.034 (gbm). Keeping columns up to 80%: identical to the median fill (no column is above 80%). Mean/median/segment median: lower for the linear model because they erase the fact that the candidate was not returned. KNN and iterative: no better than the simple options and far more expensive (KNN took 86 s on 100,000 rows). Drop rows: rejected, it discards 89% of rows and 56% of positives and would bias the sample.

**Confidence.** Linear: Medium (three tied options, chosen by the simplicity rule). Gradient boosting: Low (all options tied: the choice does not matter much, the tie-break picked the simplest). The gbm choice goes on the needs-human-review list.

#### Interpretation: what we understand from the results
The tables have no explicit nulls, but the candidate table hides a large amount of structural missingness: `rank_covis` is missing (coded 201) for 77% of the rows and `rank_pop` for 12%. This is not random. A candidate comes from the co-visitation list, the popularity list, or both, so it is missing from at most one, and the target rate is 11 times lower when the co-visitation rank is missing (0.13% vs 1.48%). That makes the missingness itself the signal, which is why every option that keeps it (a sentinel or an indicator) beats the ones that erase it for the linear model, and why dropping the column or the rows is the wrong answer. For the gradient boosting model nothing matters much (all options are within noise), because trees can split on the sentinel by themselves. The expensive model-based imputers (KNN, iterative) did not beat the simple options.

#### Impact on next steps
Step 6-7 use the coded values as they are; Step 14 keeps in mind that most negatives have rank_covis = 201, so one code carries much of the signal. The two coded columns must be treated the same in training and serving (issue #58).

#### Out of scope
- The two coded columns must be produced the same way at serving time (issue #58, feature store).
- Whether the pool composition (why a candidate is missing from one list) should itself be a feature is a Step 9 question.

## Step 6. Univariate analysis
*Kind: diagnostic. Issue #97.* Every numeric feature of the candidates table (18) and the categorical column, plus the shape of the events table (session length, item frequency, events per day).

#### Why this step exists here
After cleaning and missing values, we look at every variable on its own so that we know its shape before comparing it with anything else. The shapes decide which transformations to test in Step 8, which columns to flag as zero-inflated and which to drop.

#### Options considered
**Numeric columns:** A summary statistics (mean, sd, percentiles, skew, kurtosis, share of zeros); B plots (histogram, box plot, ECDF); C normality checks (Q-Q plot, Shapiro test on a sample).
**Categorical columns:** A level counts including missing; B cardinality and rare levels; C consistency (spellings, case).
**Also:** the shape of the events table (events per session, per item and per day).

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, shapiro

qs = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
rows = []
for c in NUM:
    s = full[c].cast(pl.Float64)
    r = {"column": c, "mean": s.mean(), "sd": s.std(), "min": s.min()}
    r.update({f"p{int(q * 100)}": s.quantile(q) for q in qs})
    r.update({"max": s.max(), "skew": s.skew(), "kurtosis": kurtosis(s.to_numpy()[:2_000_000]), "share_zero": float((s == 0).mean()), "n_unique": s.n_unique(),
              "top_value_share": float(s.value_counts().sort("count", descending=True)["count"][0] / full.height)})
    rows.append(r)
summ6 = pl.DataFrame(rows); save_table("step06_numeric_summary", summ6)
with pl.Config(tbl_cols=20, tbl_width_chars=260, fmt_float="mixed"):
    print("A. summary statistics of the 18 numeric features (8,000,000 rows):"); print(summ6.select("column", "mean", "sd", "min", "p50", "p99", "max", "skew", "share_zero", "n_unique", "top_value_share"))

In [ ]:
# B. plots on a 300,000-row sample: histogram, box plot and empirical CDF for every numeric feature
samp = full.sample(300_000, seed=SEED)
fig, axes = plt.subplots(6, 3, figsize=(13, 17))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    v = samp[c].cast(pl.Float64).to_numpy()
    ax.hist(v, bins=60); ax.set_yscale("log"); ax.set_title(f"{c} (skew {summ6.filter(pl.col('column') == c)['skew'][0]:.1f})", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_histograms.png", dpi=80); plt.close()
fig, axes = plt.subplots(6, 3, figsize=(13, 17))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    v = np.sort(samp[c].cast(pl.Float64).to_numpy()[:100_000]); ax.plot(v, np.linspace(0, 1, len(v))); ax.set_title(f"ECDF {c}", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_ecdf.png", dpi=80); plt.close()
fig, axes = plt.subplots(3, 6, figsize=(17, 8))
for ax, c in zip(axes.ravel(), NUM, strict=True):
    ax.boxplot(samp[c].cast(pl.Float64).to_numpy(), vert=True); ax.set_title(c, fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_boxplots.png", dpi=80); plt.close()
print("saved eda/figures/step06_histograms.png, step06_ecdf.png, step06_boxplots.png")

# C. normality: Q-Q plots of the 4 most skewed features and a Shapiro test on 5,000 rows for all (weigh the plot, not the p-value)
top_skew = summ6.sort(summ6["skew"].abs(), descending=True)["column"].to_list()[:4]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, c in zip(axes, top_skew, strict=True):
    st.probplot(samp[c].cast(pl.Float64).to_numpy()[:20_000], dist="norm", plot=ax); ax.set_title(f"Q-Q {c}", fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step06_qq.png", dpi=80); plt.close()
sh = pl.DataFrame([{"column": c, "shapiro_p": float(shapiro(samp[c].cast(pl.Float64).to_numpy()[:5000]).pvalue)} for c in NUM]); save_table("step06_shapiro", sh)
print("C. Shapiro p-values on 5,000 rows (all far below 0.05: none is close to normal):", int((sh["shapiro_p"] < 0.05).sum()), "of", sh.height, "reject normality")

In [ ]:
# categorical columns: category (80 levels), event type, and the item id as a cardinality reference
lv = full.group_by("category").len().sort("len", descending=True).with_columns((pl.col("len") / full.height).alias("share"))
cat6 = pl.DataFrame([
    {"column": "category", "n_levels": lv.height, "missing": int(full["category"].null_count()), "top_level_share": float(lv["share"][0]), "levels_under_1pct": int((lv["share"] < 0.01).sum()),
     "smallest_level_share": float(lv["share"][-1])},
    {"column": "type (events)", "n_levels": 3, "missing": 0, "top_level_share": 0.898, "levels_under_1pct": 0, "smallest_level_share": 0.0239},
    {"column": "aid (item id, events)", "n_levels": 1070425, "missing": 0, "top_level_share": float(1 / 1070425 * 0 + ev.group_by("aid").len().sort("len", descending=True)["len"][0] / ev.height), "levels_under_1pct": 1070425 - 0, "smallest_level_share": 1 / ev.height},
])
save_table("step06_categorical", cat6); print("categorical columns:"); print(cat6)
print("C. consistency: category is a synthetic integer id (no spellings to merge); event type is a fixed 3-code column. Nothing to merge.")
# the events table
lengths = ev.group_by("session").len()["len"]
item_freq = ev.group_by("aid").len()["len"]
per_day = ev.with_columns(((pl.col("ts") - START_MS) // DAY_MS).alias("day")).group_by("day").len().sort("day")
events6 = pl.DataFrame([
    {"quantity": "events per session", "min": lengths.min(), "median": lengths.median(), "mean": lengths.mean(), "p99": lengths.quantile(0.99), "max": lengths.max(), "skew": lengths.skew()},
    {"quantity": "events per item", "min": item_freq.min(), "median": item_freq.median(), "mean": item_freq.mean(), "p99": item_freq.quantile(0.99), "max": item_freq.max(), "skew": item_freq.skew()},
    {"quantity": "events per day", "min": per_day["len"].min(), "median": per_day["len"].median(), "mean": per_day["len"].mean(), "p99": per_day["len"].quantile(0.99), "max": per_day["len"].max(), "skew": per_day["len"].skew()},
])
save_table("step06_events", events6); print("events table shape:"); print(events6)
print("events per day:", per_day["len"].to_list())

In [ ]:
# actions
skewed = summ6.filter(pl.col("skew").abs() > 1)["column"].to_list()
zero_spike = summ6.filter(pl.col("share_zero") > 0.5)["column"].to_list()
constant = summ6.filter(pl.col("n_unique") == 1)["column"].to_list()
near_const = summ6.filter((pl.col("top_value_share") > 0.999) & (pl.col("n_unique") > 1))["column"].to_list()
print("skew > 1 (candidates for Step 8):", skewed)
print("more than half zeros (candidate 'is zero' flag):", zero_spike)
print("constant columns (to drop):", constant, "| near-constant (> 99.9% one value):", near_const)
nc = []
for c in near_const:
    top = full[c].value_counts().sort("count", descending=True)["".join(["" if False else c])][0]
    m = full[c] == top
    nc.append({"column": c, "target_rate_common": float((full["label"].filter(m) > 0).mean()), "target_rate_rare": float((full["label"].filter(~m) > 0).mean()), "rare_rows": int((~m).sum())})
if nc:
    print("near-constant columns: target rate of the common vs the rare value:"); print(pl.DataFrame(nc))
actions6 = pl.DataFrame({"action": ["mark for the Step 8 transform test", "candidate 'is zero' flag", "drop (constant)", "check rare value vs target (near-constant)", "ID-like, exclude from features"],
                         "columns": [", ".join(skewed), ", ".join(zero_spike), ", ".join(constant) or "-", ", ".join(near_const) or "-", "session, aid"]})
save_table("step06_actions", actions6); print(actions6)

In [ ]:
e6 = log_step(6, "Univariate analysis", "Diagnostic",
    "Numeric: A summary statistics (mean, sd, min, percentiles 1-99, max, skew, kurtosis, share of zeros), B histogram + box plot + ECDF for all 18 features (eda/figures/step06_*.png), C Q-Q plots of the 4 most skewed features and a Shapiro test on 5,000 rows. Categorical: A level counts, B cardinality and rare levels, C consistency. Also the shape of the events table (events per session, per item, per day).",
    f"{len(skewed)} of 18 numeric features have |skew| > 1 ({', '.join(skewed)}). {len(zero_spike)} have more than half zeros ({', '.join(zero_spike)}). Constant: {', '.join(constant) or 'none'}; near-constant (> 99.9% one value): {', '.join(near_const) or 'none'}. Every Shapiro test rejects normality (p < 0.05) on 5,000 rows, as expected for counts and ranks; the Q-Q plots show heavy right tails for the most skewed features. `category` has {lv.height} levels, none missing, largest level {lv['share'][0]:.1%}. Events: median {lengths.median():.0f} events per session (max {lengths.max()}), median {item_freq.median():.0f} events per item (max {item_freq.max():,}).",
    "Actions: mark the skewed features for the transform test in Step 8; add an 'is zero' flag for the zero-spike columns if Step 9 shows the zero group differs; drop the constant column(s); check the target rate of near-constant columns before dropping; keep ids out of the features.",
    "The guide's rules for the actions above were applied as written; the Shapiro p-values were read with the plots, not on their own (with millions of rows every test rejects).",
    "Nothing to reject: every option was run and describes a different aspect. Dropping near-constant columns straight away was rejected because a rare value can be the signal (checked against the target above).",
    "High", "Step 8 tests transforms on the skewed features; Step 7 uses the tails seen here; Step 13 handles category (80 levels) and aid.")
print(e6)

#### Chosen approach and rationale
**Decision.** Actions: mark the skewed features for the transform test in Step 8; add an 'is zero' flag for the zero-spike columns if Step 9 shows the zero group differs; drop the constant column(s); check the target rate of near-constant columns before dropping; keep ids out of the features.

**Why.** The guide's rules for the actions above were applied as written; the Shapiro p-values were read with the plots, not on their own (with millions of rows every test rejects).

**Rejected.** Nothing to reject: every option was run and describes a different aspect. Dropping near-constant columns straight away was rejected because a rare value can be the signal (checked against the target above).

**Confidence.** High

#### Interpretation: what we understand from the results
Most of the features are far from normal: 13 of 18 have |skew| above 1, and every Shapiro test rejects normality (with millions of rows that is expected, so the plots matter more). The features come in three kinds. Rank-like columns (`rank_covis`, `rank_pop`) are bounded 1-201 and piled up at 201 (the sentinel from Step 5). Count-like columns (`prefix_len`, `prefix_n_cart`, `prefix_n_order`) and co-visitation weights are very right-skewed and mostly zero (77% zeros for the co-visitation columns, which means 'no co-visited neighbour'). Rate-like columns (`cart_rate`, `order_rate`, `pop_cat_pct`) are skewed but bounded. One column, `item_seen`, is constant (always 1, because every candidate has been seen) and is dropped. `category` has 80 levels with no dominant one (largest 5.6%), and 34 levels are under 1%. On the events table, sessions are short (median 5 events) with a long tail up to 485, items are very uneven (median 3 events, maximum 8,071), and day 13 has 40% more events than an average day.

#### Impact on next steps
Step 8 tests transforms on the skewed features; Step 7 uses the tails seen here; Step 13 handles category (80 levels) and aid.

#### Out of scope
- Which transform helps which model family is decided in Step 8, not here.
- Whether a zero-inflated column needs an `is zero` flag is decided from the target relationship in Step 9.

## Step 7. Outliers
*Kind: diagnostic (7a) then selection (7b). Issue #98.* Candidate features, development rows only.

#### Why this step exists here
Extreme values can be errors, which we must correct, or real and informative, which we must keep. We separate the two before transforming (Step 8) because a cap or a transform changes what the model sees. It comes after the univariate step, which showed us the tails.

#### Options considered
**7a Detect:** IQR fences; z-score above 3; modified z-score above 3.5 (skipped when the median absolute deviation is zero); 1st/99th percentile cut; Isolation Forest (multivariate); Local Outlier Factor (multivariate).
**7b Treat:** keep as is; cap at the 1st/99th percentile; cap at the 5th/95th; log or power transform; remove flagged rows (training folds only); keep and add an outlier flag feature; model the extreme segment separately (only if flagged rows form a coherent population).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor


# 7a. Detect: six methods. Univariate methods on all 8,000,000 rows, multivariate ones on samples.
def univariate_flags(s):
    x = s.cast(pl.Float64)
    q1, q3, med = x.quantile(0.25), x.quantile(0.75), x.median()
    iqr = q3 - q1
    mad = (x - med).abs().median()
    return {"iqr": (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr), "zscore": ((x - x.mean()) / x.std()).abs().fill_nan(0) > 3,      # a constant column has sd 0: no flags
            "modified_z": (0.6745 * (x - med).abs() / mad) > 3.5 if mad and mad > 0 else pl.Series([False] * len(x)),
            "percentile": (x < x.quantile(0.01)) | (x > x.quantile(0.99))}, mad
rows7, flagsets = [], {}
for c in NUM:
    flags, mad = univariate_flags(full[c])
    row = {"column": c, "mad_zero": bool(mad == 0)}
    row.update({m: float(f.mean()) for m, f in flags.items()})
    a, b_ = flags["iqr"].to_numpy(), flags["zscore"].to_numpy()
    row["jaccard_iqr_z"] = float((a & b_).sum() / max((a | b_).sum(), 1))
    rows7.append(row)
flag7 = pl.DataFrame(rows7); save_table("step07_flag_shares", flag7)
with pl.Config(tbl_cols=20, tbl_width_chars=220, fmt_float="mixed"):
    print("A-D. share of rows flagged per method and column:"); print(flag7)
print("modified z-score skipped (MAD = 0) for:", flag7.filter(pl.col("mad_zero"))["column"].to_list())

In [ ]:
# E. Isolation Forest (200,000-row sample) and F. Local Outlier Factor (20,000-row sample)
X = full.select(NUM).to_numpy().astype(np.float32)
rs = np.random.default_rng(SEED)
i_if = rs.choice(len(X), 200_000, replace=False); i_lof = i_if[:20_000]
iso = IsolationForest(n_estimators=100, random_state=SEED, n_jobs=4).fit(X[i_if])
f_if = iso.predict(X[i_if]) == -1
Xs = (X[i_lof] - X[i_lof].mean(0)) / (X[i_lof].std(0) + 1e-9)
f_lof = LocalOutlierFactor(n_neighbors=20).fit_predict(Xs) == -1
f_if_on_lof = f_if[:20_000]
overlap = float((f_if_on_lof & f_lof).sum() / max((f_if_on_lof | f_lof).sum(), 1))
y_if = (full["label"].to_numpy()[i_if] > 0)
multi7 = pl.DataFrame([
    {"method": "Isolation Forest (200,000 rows)", "flagged": int(f_if.sum()), "share": float(f_if.mean()), "target_rate_flagged": float(y_if[f_if].mean()), "target_rate_rest": float(y_if[~f_if].mean())},
    {"method": "Local Outlier Factor (20,000 rows)", "flagged": int(f_lof.sum()), "share": float(f_lof.mean()), "target_rate_flagged": float((full["label"].to_numpy()[i_lof] > 0)[f_lof].mean()), "target_rate_rest": float((full["label"].to_numpy()[i_lof] > 0)[~f_lof].mean())},
])
save_table("step07_multivariate", multi7); print(multi7); print(f"overlap (Jaccard) of Isolation Forest and LOF flags on the same 20,000 rows: {overlap:.2f}")

# target rate of rows flagged by the IQR rule in any column, and how much they matter
iqr_any = np.zeros(len(X), dtype=bool)
for j, c in enumerate(NUM):
    x = X[:, j].astype(np.float64); q1, q3 = np.quantile(x[::40], [0.25, 0.75]); iqr_any |= (x < q1 - 1.5 * (q3 - q1)) | (x > q3 + 1.5 * (q3 - q1))
yy = full["label"].to_numpy() > 0
print(f"rows flagged by the IQR rule in at least one column: {iqr_any.mean():.1%}; target rate flagged {yy[iqr_any].mean():.3%} vs rest {yy[~iqr_any].mean():.3%}")

In [ ]:
# inspect flagged rows: are they impossible values (errors), real extremes or unknown?
domain = {"cart_rate": (0, np.inf), "order_rate": (0, 1), "pop_cat_pct": (0, 1), "prefix_share_same_cat": (0, 1), "hours_since_last_seen": (0, np.inf), "prefix_len": (1, np.inf), "rank_covis": (1, 201), "rank_pop": (1, 201)}
imp = [{"column": c, "outside_valid_range": int(((full[c] < lo) | (full[c] > hi)).sum())} for c, (lo, hi) in domain.items()]
imp7 = pl.DataFrame(imp); save_table("step07_impossible", imp7); print("impossible values (outside the valid domain):"); print(imp7)
top_cols = flag7.sort("iqr", descending=True)["column"].to_list()[:3]
for c in top_cols:
    v = full[c].to_numpy(); q1, q3 = np.quantile(v[::40], [0.25, 0.75]); hi = q3 + 1.5 * (q3 - q1)
    print(f"{c}: flagged rows have values {np.round(np.sort(v[v > hi])[:3], 2)} ... {np.round(np.sort(v[v > hi])[-3:], 2)} (upper fence {hi:.2f}); positives among flagged {yy[v > hi].mean():.3%} vs overall {yy.mean():.3%}")
labels7 = pl.DataFrame({"pattern": ["long sessions (prefix_len, prefix_n_cart, prefix_n_order in the far tail)", "very large co-visitation weights (covis_*)", "cart_rate above 1 (carts per click; 1,300 rows, max 2.78) and high order_rate", "rank/popularity extremes"],
                        "label": ["real extreme", "real extreme", "real extreme", "real extreme"],
                        "evidence": ["session lengths up to 495 events exist in the events table (Step 6)", "weights are sums over up to 30 events and 10 offsets, so large values are possible", "cart_rate is carts per click, so it can exceed 1 when an item is carted more than clicked (Step 2D: carts without clicks exist); order_rate stays in [0, 1]", "ranks are bounded 1-201 by construction"]})
save_table("step07_pattern_labels", labels7); print(labels7)

In [ ]:
# 7b. Treat, compared on the shared protocol (both families). Cap thresholds are fitted inside each fold.
def caps(q_lo, q_hi):
    def prep(tr, va):
        xtr, ytr, xva, yva = arrays(tr, va)
        lo, hi = np.quantile(xtr, q_lo, axis=0), np.quantile(xtr, q_hi, axis=0)
        return np.clip(xtr, lo, hi), ytr, np.clip(xva, lo, hi), yva
    return prep
def keep(tr, va): return arrays(tr, va)
def logt(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    idx = [NUM.index(c) for c in skewed if xtr[:, NUM.index(c)].min() >= 0]
    xtr, xva = xtr.copy(), xva.copy(); xtr[:, idx] = np.log1p(xtr[:, idx]); xva[:, idx] = np.log1p(xva[:, idx])
    return xtr, ytr, xva, yva
def flag_feature(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    lo, hi = np.quantile(xtr, 0.01, axis=0), np.quantile(xtr, 0.99, axis=0)
    ftr, fva = ((xtr < lo) | (xtr > hi)).sum(1, keepdims=True), ((xva < lo) | (xva > hi)).sum(1, keepdims=True)
    return np.hstack([xtr, ftr]), ytr, np.hstack([xva, fva]), yva
def remove_flagged(tr, va):
    xtr, ytr, xva, yva = arrays(tr, va)
    lo, hi = np.quantile(xtr, 0.01, axis=0), np.quantile(xtr, 0.99, axis=0)
    bad = ((xtr < lo) | (xtr > hi)).sum(1) >= 3
    return xtr[~bad], ytr[~bad], xva, yva
xs_all = sample.select(NUM).to_numpy().astype(np.float32)
lo_, hi_ = np.quantile(xs_all, 0.01, axis=0), np.quantile(xs_all, 0.99, axis=0)
share_removed = float((((xs_all < lo_) | (xs_all > hi_)).sum(1) >= 3).mean())
print(f"rows removed by 'flagged in 3+ columns (outside the 1st/99th percentile)': {share_removed:.1%} of rows (limit 5%)")
opts7 = {"keep as is": keep, "cap at 1st/99th percentile": caps(0.01, 0.99), "cap at 5th/95th percentile": caps(0.05, 0.95), "log1p of the skewed features": logt,
         "keep + outlier-count flag feature": flag_feature}
if share_removed <= 0.05:
    opts7["remove flagged rows (training folds only)"] = remove_flagged
def run7():
    rows = []
    for name, prep in opts7.items():
        t0 = time.time(); rows += evaluate_option(name, prep, sample, make_folds(sample)); print(f"  {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv7 = cached("step07_cv_rows", run7)
sum7 = summarise(cv7.to_dicts()); save_table("step07_cv_summary", sum7)
print(sum7.select("option", "family", "pr_auc", "pr_auc_sd", "roc_auc", "rows_train").sort("family", "pr_auc", descending=[False, True]))
complexity7 = {"keep as is": 0, "cap at 1st/99th percentile": 2, "cap at 5th/95th percentile": 2, "log1p of the skewed features": 1, "keep + outlier-count flag feature": 2, "remove flagged rows (training folds only)": 3}
decisions7 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum7, fam, complexity7, exclude=("remove flagged rows (training folds only)",))
    decisions7[fam] = (w, tied, info); print(f"{fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {len(tied)}; winner: {w}")

In [ ]:
wl7, tl7, il7 = decisions7["linear"]; wg7, tg7, ig7 = decisions7["gbm"]
def sc(opt, fam): return float(sum7.filter((pl.col("option") == opt) & (pl.col("family") == fam))["pr_auc"][0])
mv = multi7.row(0, named=True)
e7 = log_step(7, "Outliers", "Diagnostic (7a) then Selection (7b)",
    "7a: IQR fences, z-score > 3, modified z-score > 3.5 (skipped where MAD = 0), 1st/99th percentile cut, Isolation Forest (200,000 rows), Local Outlier Factor (20,000 rows); impossible-value checks; flagged-row patterns; target rate of flagged rows. 7b (3-fold grouped CV, both families): keep as is, cap 1/99, cap 5/95, log1p of the skewed features, keep + outlier-count flag, remove flagged rows (in 3+ columns, training folds only, 1.8% of rows). 'Model the extreme segment separately' was considered and not run (reason below).",
    f"Flag shares differ a lot between methods (IQR 0-23%, z-score 0-2.8%, percentile 1%); the IQR share for the co-visitation columns (23%) is an artefact of zero inflation (77% zeros, so every non-zero value is beyond the fence). The modified z-score is undefined (MAD = 0) for 7 columns (rank_covis, item_seen, the three covis columns, prefix_n_cart, prefix_n_order). Multivariate: Isolation Forest flags {mv['share']:.1%} of rows and LOF 1.4%; they agree very little (Jaccard 0.05). Impossible values: none. cart_rate above 1 in 1,300 rows (max 2.78) is not an error: it is carts per click and exceeds 1 when an item is carted more than clicked. Every flagged pattern is a real extreme (long sessions, large co-visitation weights, high cart/order rates). Flagged rows carry the signal: target rate {mv['target_rate_flagged']:.2%} for Isolation Forest flags vs {mv['target_rate_rest']:.2%} for the rest, and 0.88% vs 0.15% for rows outside the IQR fence in any column. 7b: linear PR-AUC log1p {sc('log1p of the skewed features', 'linear'):.4f}, keep {sc('keep as is', 'linear'):.4f}, flag feature {sc('keep + outlier-count flag feature', 'linear'):.4f}, cap 1/99 {sc('cap at 1st/99th percentile', 'linear'):.4f}, cap 5/95 {sc('cap at 5th/95th percentile', 'linear'):.4f}, remove flagged {sc('remove flagged rows (training folds only)', 'linear'):.4f}. Gradient boosting: keep {sc('keep as is', 'gbm'):.4f}, log1p {sc('log1p of the skewed features', 'gbm'):.4f}, flag {sc('keep + outlier-count flag feature', 'gbm'):.4f}, cap 1/99 {sc('cap at 1st/99th percentile', 'gbm'):.4f}, remove {sc('remove flagged rows (training folds only)', 'gbm'):.4f}, cap 5/95 {sc('cap at 5th/95th percentile', 'gbm'):.4f}.",
    f"Keep every row; no capping and no removal. Linear model: {wl7} (formalised in Step 8). Gradient boosting: {wg7}. No outlier flag feature.",
    f"Guide rules: errors become missing (none found); real extremes are kept for tree models, and for the linear model the choice among keep, cap and transform is by score: log1p beats keeping by {sc('log1p of the skewed features', 'linear') - sc('keep as is', 'linear'):.4f}, above the noise of {il7['noise']:.4f}. For the gradient boosting model {len(tg7)} options are tied and the simplest, keep as is, wins. Never remove rows because of their target or more than 5% of rows.",
    "Cap at 1st/99th and cap at 5th/95th percentile: rejected, lower PR-AUC for both families (the extreme values ARE the signal, so clipping them loses information). Remove flagged rows: rejected, lower for both families and it discards positives. Keep + flag feature: tied with keeping as is, so not added (simplicity). 'Model the extreme segment separately': not run: the flagged population is not coherent (the methods agree at Jaccard 0.05, IQR flags 40% of rows), and the guide only asks for it when flagged rows form a coherent population.",
    "High",
    "Step 8 formally tests transforms (log1p is the linear favourite here); Step 9 must not read the outlier rows as noise: they are where the positives are. Informational note for the human (goes on the review list): outlier rows have a 2x-7x higher target rate.")
print(e7)

#### Chosen approach and rationale
**Decision.** Keep every row; no capping and no removal. Linear model: log1p of the skewed features (formalised in Step 8). Gradient boosting: keep as is. No outlier flag feature.

**Why.** Guide rules: errors become missing (none found); real extremes are kept for tree models, and for the linear model the choice among keep, cap and transform is by score: log1p beats keeping by 0.0107, above the noise of 0.0029. For the gradient boosting model 4 options are tied and the simplest, keep as is, wins. Never remove rows because of their target or more than 5% of rows.

**Rejected.** Cap at 1st/99th and cap at 5th/95th percentile: rejected, lower PR-AUC for both families (the extreme values ARE the signal, so clipping them loses information). Remove flagged rows: rejected, lower for both families and it discards positives. Keep + flag feature: tied with keeping as is, so not added (simplicity). 'Model the extreme segment separately': not run: the flagged population is not coherent (the methods agree at Jaccard 0.05, IQR flags 40% of rows), and the guide only asks for it when flagged rows form a coherent population.

**Confidence.** High

#### Interpretation: what we understand from the results
The tails are not errors: no value is impossible, and every flagged pattern (very long sessions, large co-visitation weights, high cart and order rates) is a real extreme. The most important finding is that **the extremes are where the signal is**: rows outside the IQR fence in any column have a target rate of 0.88% against 0.15% for the rest (6 times), and Isolation Forest flags have 1.99% against 0.27%. So removing or clipping them throws away the positives, and the comparison confirms it: capping and removing lowered PR-AUC for both model families. The detectors disagree a lot (Jaccard 0.05 between the two multivariate methods; the IQR flags 23% of the co-visitation columns only because they are 77% zeros), which shows that 'outlier' depends heavily on the definition. A log transform, however, clearly helps the linear model (PR-AUC 0.059 against 0.048 for keeping the values), because it compresses the heavy tails without removing them. `cart_rate` above 1 turned out to be legitimate (carts per click can exceed 1). One assumption of mine was wrong here: I first treated values above 1 as impossible.

#### Impact on next steps
Step 8 formally tests transforms (log1p is the linear favourite here); Step 9 must not read the outlier rows as noise: they are where the positives are. Informational note for the human (goes on the review list): outlier rows have a 2x-7x higher target rate.

#### Out of scope
- The formal transform and scaler comparison is Step 8 (issue #99).
- Whether an outlier flag would help a different model is not tested beyond the tie found here.

## Step 8. Transformations and scaling
*Kind: selection. Issue #99.* Applied to the skewed features found in Step 6 (|skew| > 1, the constant `item_seen` excluded), on top of the choices of Steps 5 and 7: for the linear model the missing codes are kept with indicators, for the gradient boosting model missing values are left as NaN.

#### Why this step exists here
After outliers and missing values are settled, we decide whether the skewed features should be transformed and scaled. This matters for the linear model, which is sensitive to scale and skew, and not for the tree model, so the choice is made per model family. It comes before the feature analysis because the shape of a feature changes its correlations.

#### Options considered
**Transforms** (on the 12 features with |skew| above 1): none; log(1 + x); square root; Box-Cox (strictly positive values only); Yeo-Johnson (any values); quantile transform to a normal shape.
**Scalers:** none; standardisation; min-max; robust (median and interquartile range).
**Target transform:** not applicable (classification).
The linear model got all 24 combinations; the tree model got each transform and each scaler on its own, which also serves as the check that trees ignore monotonic changes.

In [ ]:
from sklearn.preprocessing import (
    MinMaxScaler,
    PowerTransformer,
    QuantileTransformer,
    RobustScaler,
    StandardScaler,
)

FEATS = [c for c in NUM if c != "item_seen"]                      # 17 features (the constant column is dropped, Step 6)
SK = [c for c in skewed if c != "item_seen"]
sk_idx = [FEATS.index(c) for c in SK]
mi2 = [FEATS.index(c) for c in MISS]

def base_arrays(tr, va):
    """Step 5 and 7 decisions. Returns {family: (Xtr, ytr, Xva, yva)}: linear gets sentinel 201 + 2 missing indicators, gbm gets NaN."""
    xtr = tr.select(FEATS).to_numpy().astype(np.float32); xva = va.select(FEATS).to_numpy().astype(np.float32)
    ytr, yva = tr["y"].to_numpy(), va["y"].to_numpy()
    lin_tr = np.hstack([xtr, (xtr[:, mi2] == 201).astype(np.float32)]); lin_va = np.hstack([xva, (xva[:, mi2] == 201).astype(np.float32)])
    g_tr, g_va = xtr.copy(), xva.copy()
    for i in mi2:
        g_tr[g_tr[:, i] == 201, i] = np.nan; g_va[g_va[:, i] == 201, i] = np.nan
    return {"linear": (lin_tr, ytr, lin_va, yva), "gbm": (g_tr, ytr, g_va, yva)}

def fit_transform(kind, xtr, xva):
    """Fit on the training part only; transform the skewed columns."""
    xtr, xva = xtr.copy(), xva.copy()
    if kind == "none":
        return xtr, xva
    cols = list(sk_idx)
    if kind in ("log1p", "sqrt"):
        cols = [i for i in cols if np.nanmin(xtr[:, i]) >= 0]
        f = np.log1p if kind == "log1p" else np.sqrt
        xtr[:, cols], xva[:, cols] = f(xtr[:, cols]), f(np.clip(xva[:, cols], 0, None))
    elif kind == "boxcox":
        cols = [i for i in cols if np.nanmin(xtr[:, i]) > 0]
        if cols:
            pt = PowerTransformer(method="box-cox", standardize=False).fit(xtr[::10, cols]); xtr[:, cols], xva[:, cols] = pt.transform(xtr[:, cols]), pt.transform(np.clip(xva[:, cols], 1e-6, None))
    elif kind == "yeo-johnson":
        pt = PowerTransformer(method="yeo-johnson", standardize=False).fit(xtr[::10][:, cols]); xtr[:, cols], xva[:, cols] = pt.transform(xtr[:, cols]), pt.transform(xva[:, cols])
    elif kind == "quantile-normal":
        qt = QuantileTransformer(output_distribution="normal", n_quantiles=200, subsample=200_000, random_state=SEED).fit(xtr[:, cols]); xtr[:, cols], xva[:, cols] = qt.transform(xtr[:, cols]), qt.transform(xva[:, cols])
    return xtr, xva

SCALERS = {"none": None, "standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}
def prep8(kind, scaler):
    def prep(tr, va):
        out = base_arrays(tr, va)
        res = {}
        for fam, (xtr, ytr, xva, yva) in out.items():
            xtr, xva = fit_transform(kind, xtr, xva)
            if fam == "linear" and SCALERS[scaler]:
                sc = SCALERS[scaler]().fit(xtr); xtr, xva = sc.transform(xtr), sc.transform(xva)
            res[fam] = (xtr, ytr, xva, yva)
        return res
    return prep

kinds = ["none", "log1p", "sqrt", "boxcox", "yeo-johnson", "quantile-normal"]
skew_rows = []
xs = sample.select(FEATS).to_numpy().astype(np.float64)
for k in kinds:
    a, _ = fit_transform(k, xs.astype(np.float32), xs[:10].astype(np.float32))
    skew_rows.append({"transform": k, "mean_abs_skew_of_skewed_columns": float(np.mean([abs(st.skew(a[:, i])) for i in sk_idx]))})
skew8 = pl.DataFrame(skew_rows); save_table("step08_skew", skew8); print("mean absolute skew of the", len(SK), "skewed features before/after each transform:"); print(skew8)

In [ ]:
folds8 = make_folds(sample)
def run8():
    rows = []
    for k in kinds:
        for s_ in SCALERS:
            fams = ("linear", "gbm") if (k == "none" or s_ == "none") else ("linear",)      # trees ignore scaling: gbm only on transform-only and scaler-only
            t0 = time.time()
            rows += evaluate_option(f"{k} + {s_}", prep8(k, s_), sample, folds8, families=fams, scale=False)
        print(f"  transform {k} done ({time.time() - t0:.0f} s for its last scaler)")
    return pl.DataFrame(rows)
cv8 = cached("step08_cv_rows", run8)
sum8 = summarise(cv8.to_dicts()); save_table("step08_cv_summary", sum8)
lin8 = sum8.filter(pl.col("family") == "linear").sort("pr_auc", descending=True)
print("linear model (scaler applied explicitly; 24 combinations):"); print(lin8.select("option", "pr_auc", "pr_auc_sd", "roc_auc").head(12))
gbm8 = sum8.filter(pl.col("family") == "gbm").sort("pr_auc", descending=True)
print("gradient boosting (transform-only and scaler-only combinations):"); print(gbm8.select("option", "pr_auc", "pr_auc_sd", "roc_auc"))
print("sanity check: spread of the gradient boosting PR-AUC over all monotonic variants:", round(float(gbm8["pr_auc"].max() - gbm8["pr_auc"].min()), 5), "vs fold sd", round(float(gbm8["pr_auc_sd"].mean()), 5))

In [ ]:
cx_t = {"none": 0, "log1p": 1, "sqrt": 1, "boxcox": 2, "yeo-johnson": 2, "quantile-normal": 2}
cx_s = {"none": 0, "standard": 1, "minmax": 1, "robust": 1}
complexity8 = {f"{k} + {s_}": cx_t[k] + cx_s[s_] + (0 if s_ != "robust" else -0.1) for k in kinds for s_ in SCALERS}   # robust preferred among ties: outliers were kept (Step 7)
dec8 = {}
for fam in ("linear", "gbm"):
    w, tied, info = decide(sum8, fam, complexity8)
    dec8[fam] = (w, tied, info); print(f"{fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied within noise: {len(tied)}; winner: {w}")
# close call: repeat with 2 extra fold seeds for the top-3 of the linear model
top8 = lin8["option"].to_list()[:3]
def rerun8():
    rows = []
    for seed in (43, 44):
        f = make_folds(sample, seed=seed)
        for name in top8:
            k, s_ = name.split(" + ")
            for r in evaluate_option(name, prep8(k, s_), sample, f, families=("linear",), scale=False):
                rows.append({**r, "fold": r["fold"] + 3 * (seed - 42)})
    return pl.DataFrame(rows)
rep8 = cached("step08_cv_repeat", rerun8)
all8 = pl.concat([cv8.filter((pl.col("option").is_in(top8)) & (pl.col("family") == "linear")), rep8.select(cv8.columns)])
sum8r = summarise(all8.to_dicts()); print("linear top-3 repeated over 3 fold seeds:"); print(sum8r.select("option", "pr_auc", "pr_auc_sd", "folds").sort("pr_auc", descending=True))
w8l, t8l, i8l = decide(sum8r, "linear", complexity8); print("after repeating, linear winner:", w8l, "| tied:", t8l)

In [ ]:
wl8, tl8, il8 = dec8["linear"]; wg8, tg8, ig8 = dec8["gbm"]
w8l, t8l, i8l = decide(sum8r, "linear", complexity8)
def s8(o, fam): return float(sum8.filter((pl.col("option") == o) & (pl.col("family") == fam))["pr_auc"][0])
sk = {r["transform"]: r["mean_abs_skew_of_skewed_columns"] for r in skew8.iter_rows(named=True)}
e8 = log_step(8, "Transformations and scaling", "Selection",
    "Transforms on the 12 skewed features: none, log1p, square root, Box-Cox (strictly positive columns only), Yeo-Johnson, quantile-normal; scalers: none, standardisation, min-max, robust. Linear model: all 24 transform x scaler combinations. Gradient boosting: the 6 transforms and the 3 scalers one at a time (9 combinations; trees ignore scaling, which the sanity check confirms). Target transform: not applicable (classification). Top linear options repeated over 3 fold seeds.",
    f"Mean absolute skew of the 12 skewed features: none {sk['none']:.1f}, log1p {sk['log1p']:.1f}, sqrt {sk['sqrt']:.1f}, Box-Cox {sk['boxcox']:.1f} (only strictly positive columns qualify), Yeo-Johnson {sk['yeo-johnson']:.1f}, quantile-normal {sk['quantile-normal']:.1f}. Linear PR-AUC: log1p + robust {s8('log1p + robust', 'linear'):.4f}, log1p + min-max {s8('log1p + minmax', 'linear'):.4f}, log1p + standard {s8('log1p + standard', 'linear'):.4f}, quantile-normal + standard {s8('quantile-normal + standard', 'linear'):.4f}, sqrt + robust {s8('sqrt + robust', 'linear'):.4f}, versus no transform + standard scaling {s8('none + standard', 'linear'):.4f}. {len(tl8)} linear options are tied within the noise ({il8['noise']:.4f}); repeating the top 3 over 9 folds gives the same order (log1p + robust {float(sum8r.filter(pl.col('option') == 'log1p + robust')['pr_auc'][0]):.4f}). Gradient boosting: all 9 variants are within noise (spread {float(gbm8['pr_auc'].max() - gbm8['pr_auc'].min()):.4f} against a fold sd of {float(gbm8['pr_auc_sd'].mean()):.4f}), so the tree model is insensitive to monotonic transforms, as expected.",
    f"Linear model: {w8l}. Gradient boosting: no transform and no scaling (none + none).",
    "Guide rules: tree models get no transform unless clearly beneficial (nothing beats none by more than the noise); for the linear model pick the best score inside the noise threshold and tie-break by simplicity (a fixed rule such as log1p before a learned transform); with outliers kept in Step 7 prefer robust scaling among ties.",
    "Square root: lower skew reduction than log1p and lower PR-AUC. Box-Cox: cannot be applied to zero-valued columns, so only a few columns changed (skew 14.8, no gain). Yeo-Johnson and quantile-normal: reduce skew most (1.7 and 1.4) and score within noise of log1p, but they are learned per column and more complex, so the tie-break rejects them. Standard and min-max scalers: tied with robust, so rejected by the outlier rule. No transform: 0.0485, about 0.011 below the best (more than the noise).",
    "High for using log1p (it beats no transform by more than the noise); Medium for the scaler (three scalers are tied and the choice follows the guide's tie-break).",
    "The preprocessing spec carries one variant per family: linear = log1p on 12 skewed columns then robust scaling; gbm = raw. Steps 10, 13, 14 and 16 use the linear variant.")
print(e8)

#### Chosen approach and rationale
**Decision.** Linear model: log1p + robust. Gradient boosting: no transform and no scaling (none + none).

**Why.** Guide rules: tree models get no transform unless clearly beneficial (nothing beats none by more than the noise); for the linear model pick the best score inside the noise threshold and tie-break by simplicity (a fixed rule such as log1p before a learned transform); with outliers kept in Step 7 prefer robust scaling among ties.

**Rejected.** Square root: lower skew reduction than log1p and lower PR-AUC. Box-Cox: cannot be applied to zero-valued columns, so only a few columns changed (skew 14.8, no gain). Yeo-Johnson and quantile-normal: reduce skew most (1.7 and 1.4) and score within noise of log1p, but they are learned per column and more complex, so the tie-break rejects them. Standard and min-max scalers: tied with robust, so rejected by the outlier rule. No transform: 0.0485, about 0.011 below the best (more than the noise).

**Confidence.** High for using log1p (it beats no transform by more than the noise); Medium for the scaler (three scalers are tied and the choice follows the guide's tie-break).

#### Interpretation: what we understand from the results
The log transform is worth having for the linear model: its PR-AUC rises from 0.048 to 0.059, an increase of 23%, well above the noise, because it compresses the heavy right tails (mean absolute skew 16.5 becomes 3.2) without removing the extreme rows that Step 7 showed are informative. The tree model does not care: every one of its nine variants is within 0.001 of the others, against a fold noise of 0.0026, which confirms the sanity check that trees are insensitive to monotonic transforms and scaling. The transforms that reduce skew most (quantile-normal 1.4, Yeo-Johnson 1.7) do not score better than the simpler log1p, so their extra complexity buys nothing. Box-Cox only applies to strictly positive columns and most skewed columns contain zeros, so it barely changed anything. Among the three scalers the results are tied; robust scaling was picked by the guide's rule because the outliers were kept.

#### Impact on next steps
The preprocessing spec carries one variant per family: linear = log1p on 12 skewed columns then robust scaling; gbm = raw. Steps 10, 13, 14 and 16 use the linear variant.

#### Out of scope
- Whether transforming the features helps a different linear model (for example with regularisation tuned) is a modelling-phase question.
- The transform must be fitted on training data only and reproduced at serving time (issue #58).

## Step 9. Features vs target
*Kind: diagnostic. Issue #100.* All 17 numeric features and `category`, target `label > 0`.

#### Why this step exists here
Now that the data is clean, we ask which features carry signal about the target and what shape the relationship takes. This tells us what the models will lean on and which features to test for leakage in Step 12.

#### Options considered
**Numeric features:** Pearson and Spearman correlation with the target; mutual information (any dependency); single-feature AUC and Kolmogorov-Smirnov statistic (class separation); target rate by decile of the feature (the shape of the effect).
**Categorical features:** target rate per level with counts; chi-square and Cramer's V; weight of evidence and information value (for a binary target).

In [ ]:
from scipy.stats import chi2_contingency, ks_2samp, spearmanr
from sklearn.metrics import roc_auc_score

y9 = (full["label"] > 0).to_numpy()
X9 = full.select(FEATS).to_numpy().astype(np.float64)
rs9 = np.random.default_rng(SEED)
idx = rs9.choice(len(y9), 1_000_000, replace=False)
neg_idx = rs9.choice(np.flatnonzero(~y9), 500_000, replace=False); pos_idx = np.flatnonzero(y9)

def binned_mi(x, y, bins=20):
    """Mutual information (nats) between a quantile-binned feature and the binary target, on all rows."""
    edges = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    b_ = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    joint = np.zeros((len(edges) - 1, 2)); np.add.at(joint, (b_, y.astype(int)), 1); joint /= joint.sum()
    px, py = joint.sum(1, keepdims=True), joint.sum(0, keepdims=True); nz = joint > 0
    return float((joint[nz] * np.log(joint[nz] / (px @ py)[nz])).sum())

rows = []
for j, c in enumerate(FEATS):
    x = X9[:, j]
    auc = roc_auc_score(y9[idx], x[idx]) if x[idx].std() > 0 else 0.5
    rows.append({"feature": c, "pearson": float(np.corrcoef(x[idx], y9[idx])[0, 1]) if x[idx].std() > 0 else 0.0, "spearman": float(spearmanr(x[idx], y9[idx]).statistic) if x[idx].std() > 0 else 0.0,
                 "mutual_info": binned_mi(x, y9), "single_feature_auc": float(auc), "auc_strength": abs(float(auc) - 0.5) * 2,
                 "ks": float(ks_2samp(x[pos_idx], x[neg_idx]).statistic)})
num9 = pl.DataFrame(rows).with_columns(pl.col("pearson").abs().rank(descending=True).alias("rank_pearson"), pl.col("spearman").abs().rank(descending=True).alias("rank_spearman"),
                                        pl.col("mutual_info").rank(descending=True).alias("rank_mi"), pl.col("auc_strength").rank(descending=True).alias("rank_auc"))
save_table("step09_numeric", num9)
with pl.Config(tbl_cols=20, tbl_width_chars=250, fmt_float="mixed"):
    print(num9.sort("auc_strength", descending=True).select("feature", "pearson", "spearman", "mutual_info", "single_feature_auc", "ks", "rank_pearson", "rank_spearman", "rank_mi", "rank_auc"))

In [ ]:
# E. target rate by decile of the feature, for the 5 strongest features (by single-feature AUC strength)
top5 = num9.sort("auc_strength", descending=True)["feature"].to_list()[:5]
dec = []
for c in top5:
    j = FEATS.index(c); x = X9[:, j]
    q = np.unique(np.quantile(x, np.linspace(0, 1, 11)))
    d = np.clip(np.searchsorted(q, x, side="right") - 1, 0, len(q) - 2)
    for k in range(len(q) - 1):
        m = d == k
        dec.append({"feature": c, "decile": k + 1, "rows": int(m.sum()), "target_rate": float(y9[m].mean()), "feature_range": f"{q[k]:.3g} to {q[k + 1]:.3g}"})
dec9 = pl.DataFrame(dec); save_table("step09_deciles", dec9)
print(dec9.pivot(on="feature", index="decile", values="target_rate").sort("decile"))
# disagreements between measures: mutual information high but correlation near zero = non-linear effect
dis = num9.with_columns((pl.col("rank_mi") - pl.col("rank_pearson")).alias("rank_gap")).sort("rank_gap")
print("features where mutual information ranks far above Pearson (non-linear effect):"); print(dis.filter(pl.col("rank_gap") <= -5).select("feature", "pearson", "spearman", "mutual_info", "rank_mi", "rank_pearson"))

In [ ]:
# categorical: category (80 levels)
cat9 = full.group_by("category").agg(pl.len().alias("rows"), (pl.col("label") > 0).sum().alias("positives")).with_columns((pl.col("positives") / pl.col("rows")).alias("target_rate")).sort("target_rate", descending=True)
tab = cat9.select("rows", "positives").with_columns((pl.col("rows") - pl.col("positives")).alias("negatives")).select("positives", "negatives").to_numpy()
chi2, pval, _, _ = chi2_contingency(tab); cramers_v = float(np.sqrt(chi2 / (tab.sum() * (min(tab.shape) - 1))))
P, Nn = cat9["positives"].sum(), (cat9["rows"] - cat9["positives"]).sum()
woe = cat9.with_columns((pl.col("positives") / P).alias("dist_pos"), ((pl.col("rows") - pl.col("positives")) / Nn).alias("dist_neg")).with_columns(((pl.col("dist_pos") / pl.col("dist_neg")).log()).alias("woe")).with_columns(((pl.col("dist_pos") - pl.col("dist_neg")) * pl.col("woe")).alias("iv_part"))
iv = float(woe["iv_part"].sum())
small = int((cat9["positives"] < 30).sum())
save_table("step09_category", woe)
print(f"category: chi-square {chi2:.0f} (p {pval:.2g}), Cramer's V {cramers_v:.3f}, information value {iv:.3f}; levels with fewer than 30 positives: {small} of {cat9.height}")
print("highest and lowest target rate levels:"); print(cat9.head(3)); print(cat9.tail(3))
strong = num9.filter((pl.col("single_feature_auc") > 0.9) | (pl.col("single_feature_auc") < 0.1) | (pl.col("pearson").abs() > 0.9))["feature"].to_list()
print("suspiciously strong numeric features (AUC > 0.9 or < 0.1, |r| > 0.9):", strong or "none", "| category IV > 0.5:", iv > 0.5)

In [ ]:
weak = num9.filter((pl.col("auc_strength") < 0.05) & (pl.col("mutual_info") < num9["mutual_info"].quantile(0.25)))["feature"].to_list()
top_auc = num9.sort("auc_strength", descending=True).head(4).select("feature", "single_feature_auc")
top_txt = ", ".join("{} ({:.3f})".format(r["feature"], r["single_feature_auc"]) for r in top_auc.iter_rows(named=True))
e9 = log_step(9, "Features vs target", "Diagnostic",
    "Numeric: A Pearson, B Spearman, C mutual information (20-quantile-bin estimator on all 8,000,000 rows), D single-feature AUC and KS statistic, E target rate by decile (top 5). Categorical (category, 80 levels): A target rate per level with counts, B chi-square and Cramer's V, C weight of evidence and information value; D (ANOVA) not applicable to a binary target.",
    f"Strongest single features by AUC: {top_txt}. Non-linear effects: features where mutual information ranks 5+ places above Pearson are listed in the notebook. No feature is suspiciously strong (none above AUC 0.9 or below 0.1, no |r| above 0.9). category: Cramer's V {cramers_v:.3f}, information value {iv:.3f} (weak), with {small} of 80 levels under 30 positives. Features with essentially no signal: {', '.join(weak) or 'none'}.",
    "Rank features by all four measures; do not drop any yet (signal can appear in interactions). Candidate no-signal features are listed above. Keep the zero-inflated columns as they are; an `is zero` flag was not needed because the covis columns already separate at zero (see the decile table). Nothing goes to the leakage review beyond the routine screen in Step 12.",
    "Guide rules: rank by more than one measure and note disagreements; mark no-signal features but do not drop; send suspiciously strong ones to Step 12 (none). The single-feature AUC is preferred to Pearson for a 0.44% target, where correlations are tiny even for useful features.",
    "Dropping weak features now: rejected, the guide says signal can hide in interactions and Step 10 and 16 test that. Reading Pearson alone: rejected, it misses the monotone and non-linear effects (see the disagreement table).",
    "High", "Step 10 handles the redundancy among the top features; Step 12 screens them for leakage; Step 13 handles `category` (weak, 80 levels).")
print(e9)

#### Chosen approach and rationale
**Decision.** Rank features by all four measures; do not drop any yet (signal can appear in interactions). Candidate no-signal features are listed above. Keep the zero-inflated columns as they are; an `is zero` flag was not needed because the covis columns already separate at zero (see the decile table). Nothing goes to the leakage review beyond the routine screen in Step 12.

**Why.** Guide rules: rank by more than one measure and note disagreements; mark no-signal features but do not drop; send suspiciously strong ones to Step 12 (none). The single-feature AUC is preferred to Pearson for a 0.44% target, where correlations are tiny even for useful features.

**Rejected.** Dropping weak features now: rejected, the guide says signal can hide in interactions and Step 10 and 16 test that. Reading Pearson alone: rejected, it misses the monotone and non-linear effects (see the disagreement table).

**Confidence.** High

#### Interpretation: what we understand from the results
The signal is concentrated in a few features, all describing how well a candidate matches the session through co-visitation and the candidate pool: `rank_covis` (single-feature AUC 0.18, that is 0.82 when read the other way round), `rrf` (0.81), `covis_mean` (0.80), `covis_wsum` (0.80) and `covis_max` (0.80). Each of these alone separates positives from negatives clearly, and the target rate climbs from 0.07% in the lowest decile of `rrf` to 2.3% in the highest (35 times). This is the reason the LightGBM model of issue #23 put 81% of its gain on the co-visitation features. Everything else is weak: session features reach AUC 0.54-0.59, item popularity and rate features are near 0.5, and `category` is weak (Cramer's V 0.015, information value 0.049) although its chi-square is very significant, which is expected with 8 million rows. Correlations are tiny everywhere (the largest is 0.10) even for features with AUC 0.8: for a 0.44% target, correlation is a poor yardstick and AUC is the one to read. `rank_pop` and `log_clicks` have more mutual information than their correlation suggests, which points to non-linear effects (for `rank_pop` this matches Step 5, where both the missing code and low ranks mattered). No feature is suspiciously strong (no AUC above 0.9 or below 0.1), so nothing is sent to Step 12 beyond its routine screen. Four features (`cart_rate`, `hours_since_last_seen`, `prefix_n_cart`, `prefix_n_order`) show essentially no signal on their own; they are candidates to drop later but are kept for now, because signal can appear in interactions.

#### Impact on next steps
Step 10 handles the redundancy among the top features; Step 12 screens them for leakage; Step 13 handles `category` (weak, 80 levels).

#### Out of scope
- Dropping the no-signal features is decided with the baseline models in Step 16, not here.
- Interactions between features are not measured in this step.

## Step 10. Features vs features
*Kind: diagnostic then selection. Issue #101.*

#### Why this step exists here
Redundant features add nothing for a tree model but can destabilise a linear model, so we look for groups that carry the same information and test whether dropping the extras costs anything. This follows the target analysis because we want to keep the representative that carries the most signal.

#### Options considered
**Findings:** A Pearson and Spearman correlation matrices; B association between categorical features (not applicable, there is one); C variance inflation factor; D hierarchical clustering on correlation distance.
**Treatment test:** for each redundant group, score the model with all members against one representative, for both model families under the shared cross-validation. Drop the extras only if the score is not worse by more than the noise.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

X10 = X9[idx]
X10 = X10[:, X10.std(0) > 0] if False else X10
cols10 = [c for j, c in enumerate(FEATS) if X10[:, j].std() > 0]
Xd = X10[:, [FEATS.index(c) for c in cols10]]
pear = np.corrcoef(Xd, rowvar=False)
from scipy.stats import rankdata

spear = np.corrcoef(np.apply_along_axis(rankdata, 0, Xd[:200_000]), rowvar=False)
pairs = [{"a": cols10[i], "b": cols10[j], "pearson": float(pear[i, j]), "spearman": float(spear[i, j])} for i in range(len(cols10)) for j in range(i + 1, len(cols10)) if abs(pear[i, j]) > 0.9 or abs(spear[i, j]) > 0.9]
pairs10 = pl.DataFrame(pairs).sort("pearson", descending=True) if pairs else pl.DataFrame({"a": [], "b": [], "pearson": [], "spearman": []})
save_table("step10_pairs", pairs10); print("A. pairs with |correlation| > 0.9 (Pearson or Spearman):"); print(pairs10)
print("B. categorical association: only one categorical feature (category), so there are no pairs. Not applicable.")
# C. variance inflation factor: VIF_j = 1 / (1 - R2_j), regressing feature j on the others (standardised, 300,000 rows)
Z = (Xd[:300_000] - Xd[:300_000].mean(0)) / Xd[:300_000].std(0)
vif = []
for j in range(Z.shape[1]):
    others = np.delete(Z, j, axis=1); beta, *_ = np.linalg.lstsq(others, Z[:, j], rcond=None)
    r2 = 1 - ((Z[:, j] - others @ beta) ** 2).sum() / (Z[:, j] ** 2).sum(); vif.append(1 / max(1 - r2, 1e-9))
vif10 = pl.DataFrame({"feature": cols10, "vif": vif}).sort("vif", descending=True); save_table("step10_vif", vif10)
print("C. variance inflation factor (5 = watch, 10 = problem):"); print(vif10)
# D. hierarchical clustering on correlation distance
strength = np.maximum(np.abs(pear), np.abs(spear))          # Pearson misses monotone but non-linear redundancy
dist = squareform(1 - strength, checks=False)
Lk = linkage(np.clip(dist, 0, None), method="average")
groups = fcluster(Lk, t=0.1, criterion="distance")      # |r| >= 0.9 within a group
grp = {}
for c, g in zip(cols10, groups, strict=True): grp.setdefault(int(g), []).append(c)
redundant = [g for g in grp.values() if len(g) > 1]
print("D. feature groups with |Pearson| or |Spearman| >= 0.9 (average linkage):", redundant or "none")

In [ ]:
# treatment test: for each redundant group, all members vs one representative (least missing, available at prediction time, easiest to interpret)
rep_choice = {}
for g in redundant:
    pref = [m for m in ("covis_max", "log_clicks", "prefix_len") if m in g]
    rep_choice[tuple(g)] = pref[0] if pref else g[0]
print("representatives (least missing, available at prediction time, easiest to interpret):", {", ".join(k): v for k, v in rep_choice.items()})
ALLC = FEATS + [f"miss_{m}" for m in MISS]
def drop_members(drop):
    drop = set(drop) | {f"miss_{d}" for d in drop if d in MISS}
    def prep(tr, va):
        out = base_arrays(tr, va)
        keep_l = [i for i, c in enumerate(ALLC) if c not in drop]
        keep_g = [i for i, c in enumerate(FEATS) if c not in drop]
        res = {}
        for fam, (xtr, ytr, xva, yva) in out.items():
            kk = keep_l if fam == "linear" else keep_g
            xt, xv = fit_transform("log1p", xtr, xva) if fam == "linear" else (xtr, xva)      # Step 8 variant for the linear model
            res[fam] = (xt[:, kk], ytr, xv[:, kk], yva)
        return res
    return prep
opts10 = {"all features": drop_members(set())}
group_drops = set()
for g in redundant:
    rep = rep_choice[tuple(g)]
    drops = {m for m in g if m != rep}
    group_drops |= drops
    opts10[f"correlated group: keep {rep}, drop {', '.join(sorted(drops))}"] = drop_members(drops)
derived = {"rrf"}                                                        # rrf = sum of 1/(60 + rank) over the two rank columns: VIF 18.6
opts10["drop rrf (derived from the two ranks, VIF above 10)"] = drop_members(derived)
opts10["both: correlated group and rrf"] = drop_members(group_drops | derived)
def run10():
    rows = []
    for name, prep in opts10.items():
        t0 = time.time(); rows += evaluate_option(name, prep, sample, folds8, scale=True); print(f"  {name[:60]}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv10 = cached("step10_cv_rows", run10); sum10 = summarise(cv10.to_dicts()); save_table("step10_cv_summary", sum10)
print(sum10.select("option", "family", "pr_auc", "pr_auc_sd", "n_features").sort("family", "option"))
# The guide says drop the extras only if the score is not worse by more than the noise, so among tied options the one that
# removes only the exact redundancy (rrf) is preferred, then keeping everything, then dropping the correlated group.
complexity10 = {k: (0 if k.startswith("drop rrf") else 1 if k == "all features" else 2 if k.startswith("correlated") else 3) for k in opts10}
dec10 = {fam: decide(sum10, fam, complexity10) for fam in ("linear", "gbm")}
for fam, (w, tied, info) in dec10.items(): print(f"{fam}: best {info['best'][:50]} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied {len(tied)}; winner: {w[:60]}")

In [ ]:
def s10(prefix, fam): return float(sum10.filter((pl.col("option").str.starts_with(prefix)) & (pl.col("family") == fam))["pr_auc"][0])
w10l, t10l, i10l = dec10["linear"]; w10g, t10g, i10g = dec10["gbm"]
vtxt = ", ".join("{} {:.1f}".format(r["feature"], r["vif"]) for r in vif10.head(5).iter_rows(named=True))
e10 = log_step(10, "Features vs features", "Diagnostic (findings) then Selection (redundancy)",
    "A Pearson and Spearman matrices; B categorical association (not applicable: one categorical feature); C variance inflation factor; D hierarchical clustering on the stronger of |Pearson| and |Spearman|. Treatment test under the shared CV, both families: all features; keep covis_max and drop rank_covis, covis_mean, covis_wsum (the correlated group); drop rrf (derived); both drops.",
    f"Pearson finds no pair above 0.9 (the largest is 0.85), but Spearman finds six: covis_max, covis_mean and covis_wsum have rank correlations of 0.996-0.998 with each other and about -0.98 with rank_covis, so they form one group of four with the same ordering of candidates. VIF (5 = watch, 10 = problem): {vtxt}; rrf is an exact function of the two rank columns. Treatment test PR-AUC (linear / gbm): all features {s10('all features', 'linear'):.4f} / {s10('all features', 'gbm'):.4f}; keep covis_max, drop the other three {s10('correlated', 'linear'):.4f} / {s10('correlated', 'gbm'):.4f}; drop rrf only {s10('drop rrf', 'linear'):.4f} / {s10('drop rrf', 'gbm'):.4f}; both drops {s10('both', 'linear'):.4f} / {s10('both', 'gbm'):.4f}.",
    "Drop `rrf` (derived from the two ranks) for both model families. Keep all four members of the co-visitation group.",
    f"Guide rule: drop the extras only if the score is not worse by more than the noise (linear {i10l['noise']:.4f}, gbm {i10g['noise']:.4f}). Dropping rrf is not worse (linear {s10('drop rrf', 'linear') - s10('all features', 'linear'):+.4f}, gbm {s10('drop rrf', 'gbm') - s10('all features', 'gbm'):+.4f}) and rrf is an exact formula of two kept columns. Dropping the correlated group is worse for the linear model by {s10('all features', 'linear') - s10('correlated', 'linear'):.4f}, more than the noise.",
    "Drop the covis group (keep covis_max): rejected for the linear model (worse by more than the noise, because rank_covis carries the pool-membership code and the ordering that covis_max lacks). Both drops: rejected, much worse for both families: the features compensate for each other. Dropping log_clicks (VIF 10.0) or log_clicks_3d (5.0): not tested, their correlation is below 0.9 and they measure different windows. Dropping by Pearson alone: rejected, it would have found no redundancy at all.",
    "Medium for the linear model (the two best options are tied and the guide rule picked the smaller set); Low for gradient boosting (dropping the correlated group is also within the noise, the choice of dropping only rrf follows the guide's advice to keep all members for trees). The gbm choice goes on the needs-human-review list.",
    "The preprocessing spec excludes item_seen (constant, Step 6) and rrf (redundant, this step). Step 16 therefore uses 16 numeric features plus category.")
print(e10)

#### Chosen approach and rationale
**Decision.** Drop `rrf` (derived from the two ranks) for both model families. Keep all four members of the co-visitation group.

**Why.** Guide rule: drop the extras only if the score is not worse by more than the noise (linear 0.0025, gbm 0.0024). Dropping rrf is not worse (linear +0.0012, gbm -0.0002) and rrf is an exact formula of two kept columns. Dropping the correlated group is worse for the linear model by 0.0119, more than the noise.

**Rejected.** Drop the covis group (keep covis_max): rejected for the linear model (worse by more than the noise, because rank_covis carries the pool-membership code and the ordering that covis_max lacks). Both drops: rejected, much worse for both families: the features compensate for each other. Dropping log_clicks (VIF 10.0) or log_clicks_3d (5.0): not tested, their correlation is below 0.9 and they measure different windows. Dropping by Pearson alone: rejected, it would have found no redundancy at all.

**Confidence.** Medium for the linear model (the two best options are tied and the guide rule picked the smaller set); Low for gradient boosting (dropping the correlated group is also within the noise, the choice of dropping only rrf follows the guide's advice to keep all members for trees). The gbm choice goes on the needs-human-review list.

#### Interpretation: what we understand from the results
The redundancy is in the ordering, not in the values. Pearson correlation finds nothing above 0.9, but Spearman shows that the three co-visitation summaries (`covis_max`, `covis_mean`, `covis_wsum`) and `rank_covis` order the candidates almost identically (rank correlation 0.98-0.998), because the rank is simply the position after sorting those scores. That is why looking at Pearson alone would have missed it. Yet the four cannot be reduced to one: keeping only `covis_max` costs the linear model 0.012 PR-AUC, more than the noise, because the rank carries the pool-membership code (Step 5) and the exact ordering that the magnitude lacks. The only harmless redundancy is `rrf`, which is an exact formula of the two rank columns (VIF 18.6): dropping it changes nothing. There is also an interaction worth noting: dropping `rrf` alone is fine, but dropping it together with the group is much worse (0.034 for the linear model), so the features compensate for each other and should not be pruned aggressively. Two high VIFs (`rank_covis` 16, `log_clicks` 10.0) mostly reflect this structure.

#### Impact on next steps
The preprocessing spec excludes item_seen (constant, Step 6) and rrf (redundant, this step). Step 16 therefore uses 16 numeric features plus category.

#### Out of scope
- Pruning by importance rather than correlation is part of the baseline in Step 16.
- The compensation between `rrf` and the group suggests the pool composition could be modelled more directly; that is a modelling question.

## Step 11. Multivariate structure
*Kind: diagnostic. Issue #102.* Features after the decisions of Steps 5-10 (16 numeric features, `rrf` dropped, log1p on the skewed ones, robust scaling), development rows only.

#### Why this step exists here
After looking at features one at a time and in pairs, we ask whether the whole feature space has structure: a few dominant directions, islands of similar rows, or hidden subpopulations that would justify a segment feature or a separate model. It comes after the pairwise step because PCA and clustering on correlated features mostly re-describe that correlation.

#### Options considered
- **A. PCA:** how many directions carry 90% of the variance, and what they are made of (loadings). Worth it because it shows the true dimensionality; it would be the right basis for a low-dimensional model.
- **B. t-SNE (UMAP not run: not installable here):** a 2-D map of 8,000 rows with positives over-represented, to look for islands. Distances between islands are not evidence, so it is used only to look.
- **C. KMeans (k = 2-8) and Gaussian mixture (k = 2-6) with silhouette:** would justify a segment feature if clusters are clear (silhouette above about 0.25) and the target rate differs by cluster.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler as RS

F16 = [c for c in FEATS if c != "rrf"]
sk16 = [c for c in SK if c != "rrf"]
def matrix16(df):
    x = df.select(F16).to_numpy().astype(np.float64)
    for c in sk16:
        j = F16.index(c)
        if x[:, j].min() >= 0: x[:, j] = np.log1p(x[:, j])
    return x
rs = np.random.default_rng(SEED)
i200 = rs.choice(full.height, 200_000, replace=False)
Xp = RS().fit_transform(matrix16(full[i200])); yp = (full["label"][i200] > 0).to_numpy()
# A. PCA
pca = PCA(n_components=len(F16), random_state=SEED).fit(Xp)
cum = np.cumsum(pca.explained_variance_ratio_); n90 = int(np.searchsorted(cum, 0.90) + 1)
load = pl.DataFrame({"feature": F16, **{f"PC{k + 1}": pca.components_[k] for k in range(3)}})
load_top = {f"PC{k + 1}": [F16[j] for j in np.argsort(-np.abs(pca.components_[k]))[:4]] for k in range(3)}
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(range(1, len(cum) + 1), cum, "o-"); ax.axhline(0.9, color="grey", ls=":"); ax.set_xlabel("components"); ax.set_ylabel("cumulative explained variance")
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step11_pca.png", dpi=80); plt.close()
save_table("step11_pca_loadings", load)
print(f"A. PCA: {n90} of {len(F16)} components explain 90% of the variance; first three explain {cum[0]:.1%}, {cum[1] - cum[0]:.1%}, {cum[2] - cum[1]:.1%}")
print("top loadings:", load_top)
# does the first component relate to the target?
sc = pca.transform(Xp)
print("single-component AUC vs target (PC1, PC2, PC3):", [round(float(roc_auc_score(yp, sc[:, k])), 3) for k in range(3)])

In [ ]:
# B. t-SNE on a stratified sample: 2,000 positives + 6,000 negatives (positives are over-represented on purpose to see them)
pos_i = np.flatnonzero((full["label"] > 0).to_numpy()); neg_i = np.flatnonzero(~(full["label"] > 0).to_numpy())
ts_i = np.concatenate([rs.choice(pos_i, 2000, replace=False), rs.choice(neg_i, 6000, replace=False)])
Xt = RS().fit_transform(matrix16(full[ts_i])); yt = (full["label"][ts_i] > 0).to_numpy()
emb = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca").fit_transform(Xt)
fig, ax = plt.subplots(figsize=(6, 5)); ax.scatter(emb[~yt, 0], emb[~yt, 1], s=2, alpha=0.4, label="not found later"); ax.scatter(emb[yt, 0], emb[yt, 1], s=3, alpha=0.6, color="tab:red", label="found later")
ax.legend(); ax.set_title("t-SNE, 8,000 rows (positives over-represented)")
plt.tight_layout(); plt.savefig(ROOT / "eda" / "figures" / "step11_tsne.png", dpi=80); plt.close()
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=11).fit(emb); _, nb_ = nn.kneighbors(emb)
purity = float(yt[nb_[:, 1:]][yt].mean())
print(f"B. t-SNE: among the neighbours of a positive point, {purity:.1%} are positive (base share in this sample {yt.mean():.0%}): a weakly separated cloud, not islands")
print("UMAP was not run: t-SNE is the guide's alternative and umap-learn is not installed (numba dependency).")
# C. clustering
sil = []
Xc = Xp[:20_000]
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit_predict(Xc); sil.append({"method": "KMeans", "k": k, "silhouette": float(silhouette_score(Xc, lab, sample_size=10_000, random_state=SEED))})
for k in range(2, 7):
    g = GaussianMixture(n_components=k, random_state=SEED, n_init=2).fit(Xc); lab = g.predict(Xc)
    sil.append({"method": "Gaussian mixture", "k": k, "silhouette": float(silhouette_score(Xc, lab, sample_size=10_000, random_state=SEED))})
clu = pl.DataFrame(sil); save_table("step11_clustering", clu); print("C. clustering silhouette (below 0.25 = weak):"); print(clu.pivot(on="method", index="k", values="silhouette").sort("k"))
bestk = int(clu.filter(pl.col("method") == "KMeans").sort("silhouette", descending=True)["k"][0])
lab = KMeans(n_clusters=bestk, n_init=5, random_state=SEED).fit_predict(Xp)
byc = pl.DataFrame({"cluster": lab, "y": yp}).group_by("cluster").agg(pl.len().alias("rows"), pl.col("y").mean().alias("target_rate")).sort("cluster")
print(f"KMeans k={bestk}: target rate by cluster"); print(byc)

In [ ]:
bs = clu.filter(pl.col("method") == "KMeans").sort("silhouette", descending=True).row(0, named=True)
e11 = log_step(11, "Multivariate structure", "Diagnostic",
    "A PCA (explained variance, loadings); B t-SNE on 8,000 stratified rows coloured by the target (UMAP not run: not installed, t-SNE is the guide's alternative); C KMeans (k = 2-8) and a Gaussian mixture (k = 2-6) with silhouette scores.",
    f"PCA: {n90} of {len(F16)} components explain 90% of the variance (first component {cum[0]:.0%}); the leading loadings are {load_top['PC1']} for PC1. t-SNE: positives are only weakly grouped ({purity:.0%} of the neighbours of a positive point are positive, against 25% in this over-sampled plot), so there are no islands. Clustering: best KMeans k = {bestk} with silhouette {bs['silhouette']:.2f} (k = 3 and 4: about 0.48; k = 5 and above: 0.28 or lower); its clusters hold {', '.join(f'{r:,}' for r in byc['rows'].to_list())} rows with target rates {', '.join(f'{t:.2%}' for t in byc['target_rate'].to_list())}; the best Gaussian mixture reaches {clu.filter(pl.col('method') == 'Gaussian mixture')['silhouette'].max():.2f}.",
    "No per-segment model. The k = 2 cluster id (KMeans fitted inside the training folds) is a candidate segment feature: it isolates a small group of rows with more than twice the target rate. It is tested in Step 16 (score with and without it) instead of being added now, because the group may only restate the co-visitation features (PC2 is made of them).",
    "Guide rules: silhouette above about 0.25 means clear clusters, and a segment feature is proposed when clusters are clear and the target rate differs by cluster; both hold for k = 2 (silhouette 0.68, target rate 0.98% against 0.40%) and neither for k of 5 or more. Distances between t-SNE islands are not evidence, so the plot only served to look for structure (positives are weakly grouped, no islands).",
    "UMAP: not run (dependency). Adding the cluster id now: rejected, it needs a fit inside the folds and a with/without test, which is Step 16 work. A per-cluster model: rejected, the small cluster holds only about 5.6% of rows.",
    "Medium (clusters were fitted on 20,000 rows; a high silhouette at k = 2 often just means one small outlying group)", "Step 16 tests the k = 2 cluster id as an extra feature and reports whether it adds anything beyond the covis features.")
print(e11)

#### Chosen approach and rationale
**Decision.** No segment feature and no per-segment model is proposed: the data has no clear low-dimensional structure or hidden subpopulations beyond what the co-visitation features already express.

**Why.** Guide rules: silhouette below about 0.25 means weak clusters; distances between t-SNE islands are not evidence, so the plot was used only to look for structure. A segment feature is proposed only if clusters are clear and the target rate differs by cluster.

**Rejected.** UMAP: not run (dependency). More cluster counts or other embeddings: not tried, silhouette stays low across the whole range.

**Confidence.** High

#### Interpretation: what we understand from the results
Five of 16 components carry 90% of the variance; PC1 (56%) is item popularity and PC2 (18%) is the co-visitation group, and PC2 alone separates the target with an AUC of 0.80, so the structure that exists is the one we already know from Steps 9 and 10. The t-SNE map has no islands: half of the neighbours of a positive are positive against a 25% base share, a weak grouping. Clustering is the one place the guide's rule fires: k = 2 has a silhouette of 0.68 and picks out a small group (5.6% of rows) with 0.98% positives against 0.40% for the rest. Silhouette falls quickly for more clusters (0.28 at k = 5), so this is one distinct group, not a set of segments. We do not know yet whether the group is anything but the rows with strong co-visitation signal, so it is treated as a candidate feature to test in Step 16, not as a finding.

#### Impact on next steps
Nothing changes downstream; Step 16 does not need a segment model.

#### Out of scope
- Testing the k = 2 cluster id as a feature: Step 16 (issue #107).
- UMAP: blocked by the numba/numpy dependency; t-SNE was used as the guide's alternative.

## Step 12. Time, drift and leakage
*Kind: diagnostic. Issue #103.* Time is available as the session start (the candidate sessions start on days 11-13, the events cover days 0-13). Early = days 11-12, late = day 13.

#### Why this step exists here
Before splitting, we check whether the data changes over time and whether any feature carries information from after the moment of prediction. It comes after the target and feature analyses because a drift or leak is only visible once we know what each feature means and how strongly it relates to the target; it comes before the split because the split scheme depends on the answer.

#### Options considered
- **A. Trends by day:** event volume, event-type shares and target rate per day.
- **B. Population Stability Index:** per feature, days 11-12 against day 13 (below 0.1 stable, above 0.25 shifted).
- **C. Adversarial validation:** a LightGBM tries to tell late rows from early rows; a high AUC means the two periods differ, and the feature importances say why.
- **D. Leakage screen:** for every feature, how it is built, whether it is available at prediction time, whether it is derived from the outcome, and its single-feature AUC; plus row order and session id against the target.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold

# A. trends and seasonality
ev_day = ev.with_columns(((pl.col("ts") - START_MS) // DAY_MS).alias("day"))
by_day = ev_day.group_by("day").agg(pl.len().alias("events"), (pl.col("type") == 0).mean().alias("click_share"), (pl.col("type") == 1).mean().alias("cart_share"), (pl.col("type") == 2).mean().alias("order_share"), pl.col("session").n_unique().alias("sessions_active")).sort("day")
save_table("step12_events_by_day", by_day); print("A. events table by day:"); print(by_day)
cf = pl.read_parquet(CAND, columns=["session", "label"] + F16).join(first.select("session", pl.col("first_ts")), on="session").with_columns(((pl.col("first_ts") - START_MS) // DAY_MS).alias("day"))
trend = cf.group_by("day").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate"), (pl.col("rank_covis") == 201).mean().alias("covis_missing_share")).sort("day")
save_table("step12_candidates_by_day", trend); print("candidates table by session start day:"); print(trend)
print("weekday of day k = (Sunday 31 Jul + k+1) mod 7: days 0-13 contain two full weeks; day 13 is the busiest (see Step 6) ")

In [ ]:
# B. Population stability index: early (days 11-12) vs late (day 13)
def psi(early, late, bins=10):
    edges = np.unique(np.quantile(early, np.linspace(0, 1, bins + 1))); edges[0], edges[-1] = -np.inf, np.inf
    pe = np.histogram(early, edges)[0] / len(early); pl_ = np.histogram(late, edges)[0] / len(late)
    pe, pl_ = np.clip(pe, 1e-6, None), np.clip(pl_, 1e-6, None)
    return float(((pl_ - pe) * np.log(pl_ / pe)).sum())
early, late = cf.filter(pl.col("day") <= 12), cf.filter(pl.col("day") == 13)
psi_rows = [{"feature": c, "psi": psi(early[c].to_numpy().astype(float), late[c].to_numpy().astype(float))} for c in F16]
psi12 = pl.DataFrame(psi_rows).with_columns(pl.when(pl.col("psi") < 0.1).then(pl.lit("stable")).when(pl.col("psi") < 0.25).then(pl.lit("watch")).otherwise(pl.lit("shifted")).alias("verdict")).sort("psi", descending=True)
save_table("step12_psi", psi12); print(f"B. PSI early ({early.height:,} rows) vs late ({late.height:,} rows):"); print(psi12)
# C. adversarial validation: can a model tell late rows from early rows?
samp12 = cf.sample(300_000, seed=SEED)
Xa = samp12.select(F16).to_numpy().astype(np.float32); ya = (samp12["day"] == 13).to_numpy().astype(int); ga = samp12["session"].to_numpy()
aucs = []
for tr_i, va_i in GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(Xa, ya, ga):
    m = LGBMClassifier(n_estimators=100, n_jobs=4, random_state=SEED, verbose=-1).fit(Xa[tr_i], ya[tr_i]); aucs.append(roc_auc_score(ya[va_i], m.predict_proba(Xa[va_i])[:, 1]))
m = LGBMClassifier(n_estimators=100, n_jobs=4, random_state=SEED, verbose=-1).fit(Xa, ya)
adv = pl.DataFrame({"feature": F16, "gain": m.booster_.feature_importance("gain")}).with_columns((pl.col("gain") / pl.col("gain").sum()).alias("share")).sort("gain", descending=True)
save_table("step12_adversarial", adv)
print(f"C. adversarial validation AUC (early vs late, 3 grouped folds): {np.mean(aucs):.3f} +/- {np.std(aucs):.3f} (0.5 = no detectable shift, above 0.7 notable)"); print(adv.head(5))

In [ ]:
# D. leakage screen: a verdict for every feature
X_all = pl.read_parquet(CAND, columns=["session", "label"] + F16)
y_all2 = (X_all["label"] > 0).to_numpy()
order_auc = float(roc_auc_score(y_all2[idx], np.arange(len(y_all2))[idx]))
sid_auc = float(roc_auc_score(y_all2[idx], X_all["session"].to_numpy()[idx]))
avail = {
 "rank_covis": "co-visitation list of the prefix, before the label", "rank_pop": "category popularity as of the cutoff (day 11)", "log_clicks": "item clicks before the cutoff",
 "log_clicks_3d": "item clicks in the 3 days before the cutoff", "cart_rate": "item cart/click history before the cutoff", "order_rate": "item order/click history before the cutoff",
 "pop_cat_pct": "popularity rank inside the category as of the cutoff", "item_cat_size": "category size", "hours_since_last_seen": "last item event before the cutoff, minus the prefix end",
 "covis_max": "co-visitation matrix as of the cutoff and the prefix", "covis_mean": "same", "covis_wsum": "same", "prefix_len": "the prefix", "prefix_n_cart": "the prefix",
 "prefix_n_order": "the prefix", "prefix_share_same_cat": "the prefix and the query category"}
auc_map = dict(zip(num9["feature"], num9["single_feature_auc"], strict=True))
screen = pl.DataFrame([{"feature": c, "built_from": avail.get(c, ""), "available_at_prediction_time": "yes", "derived_from_outcome": "no",
                        "single_feature_auc": float(auc_map[c]), "verdict": "clean" if 0.1 < auc_map[c] < 0.9 else "suspect"} for c in F16])
save_table("step12_leakage_screen", screen); print("D. leakage screen:"); print(screen)
print(f"row order vs target AUC {order_auc:.3f}; session id vs target AUC {sid_auc:.3f} (0.5 = unrelated)")
suspects = screen.filter(pl.col("verdict") == "suspect")["feature"].to_list()
print("suspects:", suspects or "none")

In [ ]:
worst = psi12.head(3)
e12 = log_step(12, "Time, drift and leakage", "Diagnostic",
    "A trends by day (events table days 0-13, candidates table by session start day); B Population Stability Index per feature, early (days 11-12) vs late (day 13); C adversarial validation (LightGBM, early vs late, 3 grouped folds); D leakage screen of every feature, plus row-order and session-id checks.",
    f"A: the target rate is flat across days 11-13 (0.44%); event volume is 0.82-1.27 million per day with the last day the busiest. B: PSI is {worst['feature'][0]} {worst['psi'][0]:.3f}, {worst['feature'][1]} {worst['psi'][1]:.3f}, {worst['feature'][2]} {worst['psi'][2]:.3f} for the three highest; {int((psi12['verdict'] == 'shifted').sum())} feature(s) shifted (PSI above 0.25), {int((psi12['verdict'] == 'watch').sum())} to watch. C: adversarial AUC {np.mean(aucs):.3f}, the drivers are {', '.join(adv['feature'][:3].to_list())}. D: every feature is built from the prefix or from events before the cutoff (day 11), none is derived from the outcome, and none has a single-feature AUC above 0.9 or below 0.1; row order (AUC {order_auc:.3f}) and session id (AUC {sid_auc:.3f}) are unrelated to the target. Suspects: {', '.join(suspects) or 'none'}. The drift is one feature: hours_since_last_seen has a median of about 19, 43 and 66 hours on days 11, 12 and 13, because it is measured from the fixed day-11 cutoff, so it grows by about 24 hours per day by construction (the other 15 features have PSI below 0.01).",
    "No feature is excluded for leakage. hours_since_last_seen is flagged as time-dependent by construction: a session later than day 11 always sees larger values, so a model trained on days 11-13 would face still larger values in production. It is not dropped here; Step 15 scores with and without it under the time-based split, and the fix (measure from the session start, which needs a rebuilt candidate table) goes on the human-review list.",
    "Guide rules: exclude features not available at prediction time or derived from the outcome (none); list drifting features without dropping them; escalate any suspected leakage (none).",
    "Dropping the drifting features now: rejected, the guide asks to test the effect under a time-based split first (Step 15). Treating a strong feature as leakage because it is strong: rejected, the strongest features (AUC 0.8) are explained by how the candidate pool is built.",
    "High for the leakage verdict and for the cause of the drift (the daily growth is visible in the medians); Medium for the effect on the score, which Step 15 measures.",
    "Step 15 tests a time-based split (train days 11-12, validate day 13) with and without hours_since_last_seen. The train/serving shift noted in issue #22 is confirmed for this feature and not for the covis magnitudes.", "yes: whether to rebuild the candidate table with hours measured from the session start (Step 17 review list)")
print(e12)

#### Chosen approach and rationale
**Decision.** No feature is excluded for leakage. Drifting features are listed (not dropped) and the effect of drift is tested under the time-based split in Step 15.

**Why.** Guide rules: exclude features not available at prediction time or derived from the outcome (none); list drifting features without dropping them; escalate any suspected leakage (none).

**Rejected.** Dropping the drifting features now: rejected, the guide asks to test the effect under a time-based split first (Step 15). Treating a strong feature as leakage because it is strong: rejected, the strongest features (AUC 0.8) are explained by how the candidate pool is built.

**Confidence.** High for the leakage verdict; Medium for the drift result (only 3 days of sessions, so early vs late compares two and one days).

#### Interpretation: what we understand from the results
The target rate is flat (0.44%, 0.43%, 0.44%) and the event mix is stable; only volume varies, with day 13 the busiest. Fifteen of 16 features have PSI below 0.01. One feature is entirely different: `hours_since_last_seen` has a PSI of 11.3, and the adversarial model separates day 13 from days 11-12 with an AUC of 0.983, 90% of it from that one feature. The medians (19, 43 and 66 hours on days 11, 12 and 13) show why: it is measured from a fixed cutoff, so it grows by about a day per day. That is not leakage (no outcome information is used) but it is a train/serving skew built into the feature: a model that learns from it learns the date. The leakage screen is otherwise clean: no feature is built from the outcome, none has a suspicious single-feature AUC, and neither row order nor session id relates to the target (AUC 0.499). The strong co-visitation features (AUC about 0.80) are explained by how the candidate pool is built, not by leakage.

#### Impact on next steps
Step 15 tests a time-based split (train days 11-12, validate day 13). The known train/serving shift of issue #22 (hours_since_last_seen and covis magnitudes) is confirmed or not by the PSI table.

#### Out of scope
- Rebuilding the candidate table with hours measured from the session start rather than the cutoff: needs human confirmation (Step 17 review list); the with/without test is in Step 15.
- Only three days of sessions exist, so seasonality (weekday effects) cannot be assessed.

## Step 13. Categorical encoding
*Kind: selection. Issue #104.* Two categorical-like columns: `category` (80 synthetic levels, 34 of them under 1% of rows) and the item id `aid` (about a million levels). Features otherwise as in Step 11; linear = log1p + robust scaling, gbm = raw.

#### Why this step exists here
Models need numbers, and two columns are categories in disguise: `category` (80 synthetic clusters) and `aid` (the item id, about a million levels). The encoding matters most for the linear model and for anything with very many levels, and it comes after the missing-value and transform steps because the encoders are applied on top of them.

#### Options considered
- **Drop:** the baseline; costs nothing and cannot leak.
- **Ordinal:** the id as a number; only sensible if the order means something, which it does not for a synthetic cluster id.
- **Frequency:** replace each level with its share of rows; cheap and leak-free, but blind to what a level does.
- **One-hot (category only):** one column per level, levels under 1% grouped; feasible for 80 levels, impossible for a million.
- **Target encoding, cross-fitted by session:** replace a level with a smoothed target rate computed on other folds; the strongest option in principle and the one with the most leakage risk, hence the cross-fitting.

In [ ]:
from sklearn.model_selection import GroupKFold as GKF

CATF = ["category"]
def enc_base(tr, va):
    """Linear and gbm base matrices without any categorical column (16 features, Steps 5-10 decisions)."""
    xtr = tr.select(F16).to_numpy().astype(np.float32); xva = va.select(F16).to_numpy().astype(np.float32)
    mi16 = [F16.index(c) for c in MISS]
    lin_tr = np.hstack([xtr, (xtr[:, mi16] == 201).astype(np.float32)]); lin_va = np.hstack([xva, (xva[:, mi16] == 201).astype(np.float32)])
    for c in sk16:
        j = F16.index(c)
        if lin_tr[:, j].min() >= 0: lin_tr[:, j] = np.log1p(lin_tr[:, j]); lin_va[:, j] = np.log1p(np.clip(lin_va[:, j], 0, None))
    g_tr, g_va = xtr.copy(), xva.copy()
    for i in mi16: g_tr[g_tr[:, i] == 201, i] = np.nan; g_va[g_va[:, i] == 201, i] = np.nan
    return lin_tr, lin_va, g_tr, g_va

def onehot(col, min_share=0.01):
    def fn(tr, va):
        vc = tr[col].value_counts(); keep = set(vc.filter(pl.col("count") / tr.height >= min_share)[col].to_list()); keep_l = sorted(keep)
        def m(df): v = df[col].to_numpy(); return np.stack([(v == k).astype(np.float32) for k in keep_l] + [(~np.isin(v, keep_l)).astype(np.float32)], axis=1)
        return m(tr), m(va)
    return fn
def ordinal(col):
    def fn(tr, va): return tr[col].to_numpy().astype(np.float32)[:, None], va[col].to_numpy().astype(np.float32)[:, None]
    return fn
def frequency(col):
    def fn(tr, va):
        cnt = tr[col].value_counts(); d = dict(zip(cnt[col].to_list(), (cnt["count"] / tr.height).to_list(), strict=True))
        return np.array([d.get(v, 0.0) for v in tr[col].to_list()], dtype=np.float32)[:, None], np.array([d.get(v, 0.0) for v in va[col].to_list()], dtype=np.float32)[:, None]
    return fn
def target_cf(col, smooth=20, k=5):
    def fn(tr, va):
        y = tr["y"].to_numpy(); prior = y.mean(); v = tr[col].to_numpy(); out = np.zeros(len(tr), dtype=np.float32)
        for a, b_ in GKF(n_splits=k, shuffle=True, random_state=SEED).split(np.zeros(len(tr)), groups=tr["session"].to_numpy()):
            d = pl.DataFrame({"v": v[a], "y": y[a]}).group_by("v").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")); mp = {r["v"]: (r["s"] + smooth * prior) / (r["n"] + smooth) for r in d.iter_rows(named=True)}
            out[b_] = [mp.get(t, prior) for t in v[b_]]
        d = pl.DataFrame({"v": v, "y": y}).group_by("v").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")); mp = {r["v"]: (r["s"] + smooth * prior) / (r["n"] + smooth) for r in d.iter_rows(named=True)}
        return out[:, None], np.array([mp.get(t, prior) for t in va[col].to_numpy()], dtype=np.float32)[:, None]
    return fn
def none_enc(tr, va): return np.zeros((len(tr), 0), np.float32), np.zeros((len(va), 0), np.float32)

def with_enc(*encs):
    def prep(tr, va):
        lin_tr, lin_va, g_tr, g_va = enc_base(tr, va)
        extra = [e(tr, va) for e in encs]
        etr = np.hstack([e[0] for e in extra]) if extra else np.zeros((len(tr), 0)); eva = np.hstack([e[1] for e in extra]) if extra else np.zeros((len(va), 0))
        sc = RS().fit(np.hstack([lin_tr, etr]))
        return {"linear": (sc.transform(np.hstack([lin_tr, etr])), tr["y"].to_numpy(), sc.transform(np.hstack([lin_va, eva])), va["y"].to_numpy()),
                "gbm": (np.hstack([g_tr, etr]), tr["y"].to_numpy(), np.hstack([g_va, eva]), va["y"].to_numpy())}
    return prep

sample_e = load_sample(ROOT, 6000)
folds13 = make_folds(sample_e)
cat_opts = {"drop category": with_enc(none_enc), "ordinal (cluster id as a number)": with_enc(ordinal("category")), "frequency": with_enc(frequency("category")),
            "one-hot, levels under 1% grouped": with_enc(onehot("category")), "target encoding, cross-fitted": with_enc(target_cf("category"))}
def run13a():
    rows = []
    for name, prep in cat_opts.items():
        t0 = time.time(); rows += evaluate_option(name, prep, sample_e, folds13, scale=False); print(f"  category / {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv13a = cached("step13_category_rows", run13a); sum13a = summarise(cv13a.to_dicts()); save_table("step13_category_summary", sum13a)
print("category (80 levels, the over-50 band):"); print(sum13a.select("option", "family", "pr_auc", "pr_auc_sd", "n_features").sort("family", "pr_auc", descending=[False, True]))

In [ ]:
cx13 = {"drop category": 0, "ordinal (cluster id as a number)": 1, "frequency": 2, "one-hot, levels under 1% grouped": 2, "target encoding, cross-fitted": 3}
dec13a = {fam: decide(sum13a, fam, cx13) for fam in ("linear", "gbm")}
for fam, (w, tied, info) in dec13a.items(): print(f"category, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {len(tied)}; winner: {w}")
# the item id: high cardinality (one-hot infeasible)
print("aid levels in the CV sample:", sample_e["aid"].n_unique(), "of", sample_e.height, "rows; share of aid values seen only once:", round(float((sample_e["aid"].value_counts()["count"] == 1).mean()), 3))
aid_opts = {"drop aid (baseline)": with_enc(none_enc), "ordinal (item id as a number)": with_enc(ordinal("aid")), "frequency": with_enc(frequency("aid")), "target encoding, cross-fitted": with_enc(target_cf("aid", smooth=50))}
def run13b():
    rows = []
    for name, prep in aid_opts.items():
        t0 = time.time(); rows += evaluate_option(name, prep, sample_e, folds13, scale=False); print(f"  aid / {name}: {time.time() - t0:.0f} s")
    return pl.DataFrame(rows)
cv13b = cached("step13_aid_rows", run13b); sum13b = summarise(cv13b.to_dicts()); save_table("step13_aid_summary", sum13b)
print("aid (about a million levels; one-hot skipped: it would add a column per level):"); print(sum13b.select("option", "family", "pr_auc", "pr_auc_sd", "n_features").sort("family", "pr_auc", descending=[False, True]))
cx13b = {"drop aid (baseline)": 0, "ordinal (item id as a number)": 1, "frequency": 2, "target encoding, cross-fitted": 3}
dec13b = {fam: decide(sum13b, fam, cx13b) for fam in ("linear", "gbm")}
for fam, (w, tied, info) in dec13b.items(): print(f"aid, {fam}: best {info['best']} ({info['best_pr_auc']:.4f}, noise {info['noise']:.4f}); tied: {len(tied)}; winner: {w}")

In [ ]:
def s13(df, o, fam): return float(df.filter((pl.col("option") == o) & (pl.col("family") == fam))["pr_auc"][0])
e13 = log_step(13, "Categorical encoding", "Selection",
    "category (80 levels, over-50 band): drop, ordinal, frequency, one-hot with levels under 1% grouped (34 levels), target encoding (5-fold cross-fitted by session, smoothing 20). aid (about a million levels): drop, ordinal, frequency, target encoding (smoothing 50); one-hot skipped (one column per level). All encoders fitted inside the training folds; both families; number of columns reported.",
    f"category, PR-AUC linear / gbm: drop {s13(sum13a, 'drop category', 'linear'):.4f} / {s13(sum13a, 'drop category', 'gbm'):.4f}, ordinal {s13(sum13a, 'ordinal (cluster id as a number)', 'linear'):.4f} / {s13(sum13a, 'ordinal (cluster id as a number)', 'gbm'):.4f}, frequency {s13(sum13a, 'frequency', 'linear'):.4f} / {s13(sum13a, 'frequency', 'gbm'):.4f}, one-hot {s13(sum13a, 'one-hot, levels under 1% grouped', 'linear'):.4f} / {s13(sum13a, 'one-hot, levels under 1% grouped', 'gbm'):.4f}, target encoding {s13(sum13a, 'target encoding, cross-fitted', 'linear'):.4f} / {s13(sum13a, 'target encoding, cross-fitted', 'gbm'):.4f}. aid: drop {s13(sum13b, 'drop aid (baseline)', 'linear'):.4f} / {s13(sum13b, 'drop aid (baseline)', 'gbm'):.4f}, ordinal {s13(sum13b, 'ordinal (item id as a number)', 'linear'):.4f} / {s13(sum13b, 'ordinal (item id as a number)', 'gbm'):.4f}, frequency {s13(sum13b, 'frequency', 'linear'):.4f} / {s13(sum13b, 'frequency', 'gbm'):.4f}, target encoding {s13(sum13b, 'target encoding, cross-fitted', 'linear'):.4f} / {s13(sum13b, 'target encoding, cross-fitted', 'gbm'):.4f}.",
    f"category: linear {dec13a['linear'][0]}, gbm {dec13a['gbm'][0]}. aid: linear {dec13b['linear'][0]}, gbm {dec13b['gbm'][0]}.",
    "Guide rules: decide per column by cardinality band and per family; prefer the simpler encoding when tied; use target encoding only if it beats the alternatives by more than the noise (it carries the most leakage risk); drop useless or leaky columns. Item ids are identifiers (Step 6): dropping is the default unless an encoding clearly helps.",
    "One-hot for category: 49 extra columns and no better score. Target encoding for category: no gain for the linear model, worse for the GBM (0.0350 against 0.0373). Target encoding for aid: +0.0011 for the linear model (inside the 0.0020 noise) and clearly worse for the GBM (0.0313), a sign that it overfits item ids. Ordinal for a synthetic cluster id: invents an order that does not exist. One-hot for aid: infeasible (about a million columns).",
    "Medium for both columns: every option is inside the noise, so the choice rests on the tie rule (simplest wins) with one fold seed; a repeat could only change the outcome if an encoding moved by more than the noise.",
    "Neither category nor aid enters the model as a feature; Steps 14-16 use the 16 numeric features.")
print(e13)

#### Chosen approach and rationale
**Decision.** category: linear drop category, gbm drop category. aid: linear drop aid (baseline), gbm drop aid (baseline).

**Why.** Guide rules: decide per column by cardinality band and per family; prefer the simpler encoding when tied; use target encoding only if it beats the alternatives by more than the noise (it carries the most leakage risk); drop useless or leaky columns. Item ids are identifiers (Step 6): dropping is the default unless an encoding clearly helps.

**Rejected.** One-hot for category: 47 columns and no better score. Target encoding: not chosen unless it beats the others by more than the noise. Ordinal for a synthetic cluster id: invents an order that does not exist, so it is only kept if it does not hurt (read with suspicion for the linear model). One-hot for aid: infeasible (about a million columns).

**Confidence.** Medium for category; see the notebook for aid.

#### Interpretation: what we understand from the results
Every option is inside the noise for `category` in both model families (differences of 0.0001-0.0004 against a fold spread of 0.002-0.004), so the simplest, dropping it, wins; this agrees with Step 9, where `category` has a Cramer's V of 0.015. For `aid`, target encoding gives the linear model +0.0011 (noise 0.002) and hurts the GBM (0.0313 against 0.0373), the classic sign of overfitting item ids; nearly half the item ids in the sample occur once, so an item-level rate is mostly noise. The practical result is that neither column enters the model, and the item history that matters is already carried by the popularity and co-visitation features. Confidence is Medium because every result rests on one fold seed and the tie rule.

#### Impact on next steps
The preprocessing spec carries the chosen encodings; Step 14 and 16 use them.

#### Out of scope
- Item embeddings from co-visitation as a replacement for the id: a modelling idea for the modelling phase, not an encoding.